# Advanced US Equity Portfolio Construction
## Classical Quant Finance + Autoencoder Latent-Factor Modeling

**Initial capital:** €1,000 · **Horizon:** 10–20 years · **Universe:** 18 US large-caps + SPY/QQQ benchmarks

---

### ⚠️ Read this before running anything

This notebook is an **educational research artifact**, not an allocation engine. Four structural caveats dominate everything below:

1. **Universe selection bias (the big one).** The 18 tickers were chosen *in 2026, knowing they were winners*. Backtesting them over 2015–2026 embeds survivorship/selection look-ahead at the universe level. The backtests **will** beat SPY — that result is an artifact of hindsight, not of the optimizers. We mitigate look-ahead *inside* the pipeline (walk-forward training, no peeking at test data), but we cannot mitigate it in the universe choice. Interpret all "alpha" accordingly.

2. **€1,000 is below the practicality threshold.** With 18 names, quarterly rebalancing, and realistic costs, a €1,000 account is better served by a single broad ETF. We model the €1,000 path anyway because the *methods* scale to any capital base.

3. **18 assets is a small cross-section for deep learning.** Autoencoders shine with hundreds of assets. With 18, PCA is a formidable baseline and may win. We report the comparison honestly either way.


4. **Synthetic data is a stress test, not a second source of truth.** Following Iñaki's suggestion, the updated notebook now generates synthetic alternate histories to test whether the AE → Black-Litterman view engine is mechanically stable. Passing those tests would not prove alpha; failing them would be a strong warning that the signal is too regime-fragile to deploy.

5. **Physics-informed risk controls are diagnostics, not magic.** The updated version adds Random Matrix Theory, correlation networks, calibrated view confidence, regime-aware tests, and a stability dashboard. These tools are there to reduce false confidence, not to claim that physics or ML can bypass noisy OOS evidence.

**Nothing here is financial advice.**


## 1. Setup

We install/import everything up front. In Google Colab, PyTorch is preinstalled; `yfinance` and `cvxpy` may need installing. All randomness is seeded for reproducibility (note: GPU nondeterminism in PyTorch can still cause tiny run-to-run differences).

In [ ]:
# In Colab, uncomment:
# !pip install -q yfinance cvxpy

import warnings, math, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import scipy.stats as st
import scipy.cluster.hierarchy as sch
from scipy.optimize import minimize, brentq
from scipy.spatial.distance import squareform

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

import cvxpy as cp

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({"figure.figsize": (11, 5), "axes.grid": True, "grid.alpha": 0.3})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {DEVICE}")

INITIAL_CAPITAL_EUR = 1_000.0
TRADING_DAYS = 252
RF_ANNUAL = 0.02          # flat risk-free assumption; replace with ^IRX if you want it time-varying
TC_BPS = 10               # one-way transaction cost, basis points of traded notional


## 2. Asset universe

18 US stocks across long-term secular themes, plus SPY and QQQ as benchmarks. We map each ticker to a sector/theme so we can audit concentration later — **theme labels are not diversification**; correlations are, and we'll see that most of these names load on the same mega-cap-growth factor.

> **Honesty check:** this universe is ~60% tech-adjacent. Calling CRWD "cybersecurity" and NVDA "semiconductors" doesn't make them independent bets. The latent-space clustering in Section 5 will make this hidden concentration visible.

In [ ]:
UNIVERSE = {
    "AAPL": "Consumer Tech",      "MSFT": "Cloud/AI",          "NVDA": "Semiconductors",
    "AMZN": "Cloud/E-commerce",   "GOOGL": "AI/Advertising",   "META": "AI/Advertising",
    "AVGO": "Semiconductors",     "AMD": "Semiconductors",     "COST": "Consumer Staples",
    "JPM":  "Financials",         "UNH": "Healthcare",         "LLY": "Healthcare",
    "V":    "Fintech/Payments",   "MA": "Fintech/Payments",    "CRWD": "Cybersecurity",
    "PANW": "Cybersecurity",      "TSLA": "Energy Transition", "NEE": "Energy Transition",
}
TICKERS = list(UNIVERSE.keys())
BENCHMARKS = ["SPY", "QQQ"]

START, END = "2015-01-01", None   # None = today

raw = yf.download(TICKERS + BENCHMARKS, start=START, end=END,
                  auto_adjust=True, progress=False)["Close"]
raw = raw.dropna(how="all")

# CRWD IPO'd June 2019 — earlier rows are NaN. We keep the full panel and
# handle the ragged start explicitly rather than silently truncating to 2019,
# which would throw away 4+ years of history for the other 17 names.
print(raw.shape)
raw.tail(3)


In [ ]:
prices = raw[TICKERS]
bench_prices = raw[BENCHMARKS]

# Daily simple returns (NaN-aware: CRWD/PANW history starts later)
rets_d = prices.pct_change()
bench_d = bench_prices.pct_change()

# Monthly returns via compounding — used for optimization (less microstructure noise)
rets_m = prices.resample("ME").last().pct_change()
bench_m = bench_prices.resample("ME").last().pct_change()

# A "complete panel" view where every asset has data (post mid-2019) — used
# wherever an estimator cannot tolerate NaNs (PCA, autoencoders, copulas).
rets_d_full = rets_d.dropna()
rets_m_full = rets_m.dropna()
print(f"Full daily panel: {rets_d_full.index[0].date()} → {rets_d_full.index[-1].date()}  ({len(rets_d_full)} days)")
print(f"Full monthly panel: {len(rets_m_full)} months")


### Core performance utilities

One set of modular, reusable functions. Every strategy in this notebook is judged by the same yardstick — no per-strategy metric tweaking.

In [ ]:
def ann_return(r, periods=TRADING_DAYS):
    r = r.dropna()
    if len(r) == 0: return np.nan
    return (1 + r).prod() ** (periods / len(r)) - 1

def ann_vol(r, periods=TRADING_DAYS):
    return r.dropna().std() * np.sqrt(periods)

def sharpe(r, rf=RF_ANNUAL, periods=TRADING_DAYS):
    er = ann_return(r, periods) - rf
    v = ann_vol(r, periods)
    return er / v if v > 0 else np.nan

def sortino(r, rf=RF_ANNUAL, periods=TRADING_DAYS):
    r = r.dropna()
    downside = r[r < 0].std() * np.sqrt(periods)
    return (ann_return(r, periods) - rf) / downside if downside > 0 else np.nan

def drawdown_series(r):
    wealth = (1 + r.fillna(0)).cumprod()
    return wealth / wealth.cummax() - 1

def max_drawdown(r):
    return drawdown_series(r).min()

def hist_cvar(r, alpha=0.95, periods=TRADING_DAYS):
    """Historical CVaR (expected shortfall) of per-period returns, reported as a positive loss."""
    r = r.dropna()
    var = np.quantile(r, 1 - alpha)
    return -r[r <= var].mean()

def perf_table(returns_dict, periods=TRADING_DAYS):
    rows = {}
    for name, r in returns_dict.items():
        rows[name] = {
            "AnnReturn": ann_return(r, periods),
            "AnnVol": ann_vol(r, periods),
            "Sharpe": sharpe(r, periods=periods),
            "Sortino": sortino(r, periods=periods),
            "MaxDD": max_drawdown(r),
            "CVaR95 (per period)": hist_cvar(r),
        }
    return pd.DataFrame(rows).T.round(3)

def herfindahl(w):
    """Concentration: 1/N (equal weight) ... 1 (single asset)."""
    w = np.asarray(w); return float(np.sum(w**2))

def sector_exposure(w, tickers=TICKERS, universe=UNIVERSE):
    s = pd.Series(w, index=tickers).groupby(lambda t: universe[t]).sum()
    return s.sort_values(ascending=False)

# Quick sanity check on benchmarks
perf_table({"SPY": bench_d["SPY"], "QQQ": bench_d["QQQ"]})


In [ ]:
# Price history (normalized to 100 at each asset's first valid date)
norm = prices / prices.bfill().iloc[0] * 100
ax = norm.plot(logy=True, alpha=0.8, linewidth=1)
ax.set_title("Price history, normalized to 100 (log scale)")
ax.legend(ncol=3, fontsize=8)
plt.show()

# Correlation matrix on the full daily panel
fig, ax = plt.subplots(figsize=(9, 7))
corr = rets_d_full.corr()
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(TICKERS))); ax.set_xticklabels(TICKERS, rotation=90, fontsize=8)
ax.set_yticks(range(len(TICKERS))); ax.set_yticklabels(TICKERS, fontsize=8)
plt.colorbar(im); ax.set_title("Daily return correlations — note how few truly low-correlation pairs exist")
plt.show()


## 3. Quant finance foundations

We implement the classical toolkit as **modular functions** all consuming `(mu, Sigma)` or raw returns, so Section 7 can swap in different covariance estimators (sample, shrunk, PCA-factor, autoencoder-factor) without rewriting optimizers.

| Method | What it fixes | What it breaks |
|---|---|---|
| Mean-variance (Markowitz) | The baseline | Hypersensitive to estimated `mu` — "error maximizer" |
| Minimum variance | Drops `mu` entirely | Concentrates in low-vol names |
| Max Sharpe | Targets risk-adjusted return | Inherits all of Markowitz's `mu` fragility |
| Ledoit–Wolf shrinkage | Noisy covariance | Still assumes stationarity |
| Black–Litterman | Anchors `mu` to market equilibrium | Garbage views → garbage posterior |
| Risk parity / HRP | Estimation error in `mu`; HRP avoids matrix inversion | Ignores expected returns by design |
| Robust optimization | Worst-case `mu` uncertainty | Can be overly conservative |
| CVaR optimization | Tail risk, non-normality | Needs many scenarios; tail estimates are noisy |

All long-only, fully invested (`w ≥ 0, Σw = 1`) — realistic for a retail long-term account.

In [ ]:
def mu_sigma(returns, periods=12):
    """Annualized mean vector and covariance matrix from (monthly) returns."""
    mu = returns.mean().values * periods
    Sigma = returns.cov().values * periods
    return mu, Sigma

def lw_sigma(returns, periods=12):
    """Ledoit-Wolf shrunk covariance: convex blend of sample cov and a structured
    target. Shrinks the extreme (noisy) eigenvalues toward the center — the single
    cheapest, highest-value fix in portfolio optimization."""
    lw = LedoitWolf().fit(returns.values)
    return lw.covariance_ * periods

def risk_parity(Sigma):
    """Equal risk contribution via SLSQP on the standard ERC objective."""
    n = Sigma.shape[0]
    def obj(w):
        port_var = w @ Sigma @ w
        rc = w * (Sigma @ w)                  # risk contributions
        return np.sum((rc - port_var / n) ** 2)
    cons = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
    res = minimize(obj, np.ones(n)/n, bounds=[(1e-5, 1)]*n, constraints=cons,
                   method="SLSQP", options={"maxiter": 1000})
    return res.x / res.x.sum()

def hrp(returns):
    """Hierarchical Risk Parity (Lopez de Prado 2016): cluster the correlation
    matrix, quasi-diagonalize, then recursively bisect allocating inverse-variance.
    No matrix inversion → robust to ill-conditioned covariances."""
    corr, cov = returns.corr(), returns.cov()
    dist = np.sqrt(0.5 * (1 - corr))
    link = sch.linkage(squareform(dist.values, checks=False), method="single")
    sort_ix = sch.leaves_list(link)
    items = corr.index[sort_ix].tolist()

    def cluster_var(cov, items):
        sub = cov.loc[items, items].values
        ivp = 1 / np.diag(sub); ivp /= ivp.sum()
        return ivp @ sub @ ivp

    w = pd.Series(1.0, index=items)
    clusters = [items]
    while clusters:
        clusters = [c[j:k] for c in clusters
                    for j, k in ((0, len(c)//2), (len(c)//2, len(c))) if len(c) > 1]
        for i in range(0, len(clusters), 2):
            if i + 1 >= len(clusters): continue
            c1, c2 = clusters[i], clusters[i+1]
            v1, v2 = cluster_var(cov, c1), cluster_var(cov, c2)
            alpha = 1 - v1 / (v1 + v2)
            w[c1] *= alpha; w[c2] *= 1 - alpha
    return w.reindex(returns.columns).values

# NOTE: solve_qp / min_variance / mean_variance / max_sharpe / black_litterman /
# cvar_optimization used to also be defined in this cell. They were removed
# because cells below (the "Transaction costs and turnover" section) redefine
# all of them with PSD-repair and multi-solver fallback -- the versions here
# were dead code, silently shadowed, never actually executed. Only the four
# functions above are NOT redefined elsewhere, so they're the only ones that
# belong in this cell.


### Transaction costs and turnover

Costs enter the notebook in two places: (1) the **walk-forward backtest** (Section 8) charges `TC_BPS` per unit of one-way turnover at every rebalance; (2) a **turnover-constrained optimizer** below caps how far a new portfolio may move from the old one — the standard production technique for keeping optimizers from churning.

At €1,000, even 10 bps round-trip with quarterly full rebalancing is ~€2–4/yr — small in % terms *only because we assume bps-based costs*. Real retail brokers charge fixed minimums (€1–5/trade), which at this account size would be **catastrophic** (18 trades × €1 × 4/yr = 7.2% annual drag). This is the single strongest argument that €1,000 belongs in one ETF.

In [ ]:
def clean_cov(Sigma, eps=1e-8):
    """
    Symmetrize covariance matrix and make it numerically PSD.
    Prevents many CVXPY / solver failures.
    """
    Sigma = np.asarray(Sigma, dtype=float)
    Sigma = np.nan_to_num(Sigma, nan=0.0, posinf=0.0, neginf=0.0)
    Sigma = 0.5 * (Sigma + Sigma.T)

    eig_min = np.linalg.eigvalsh(Sigma).min()
    if eig_min < eps:
        Sigma += np.eye(Sigma.shape[0]) * (eps - eig_min)

    return Sigma


def clean_weights(w_val, max_w=None):
    """
    Clean numerical noise from solver output.
    """
    w_val = np.asarray(w_val, dtype=float).reshape(-1)
    w_val = np.nan_to_num(w_val, nan=0.0, posinf=0.0, neginf=0.0)
    w_val = np.clip(w_val, 0, None)

    if max_w is not None:
        w_val = np.minimum(w_val, max_w)

    if w_val.sum() <= 0:
        raise ValueError("Solver returned invalid all-zero weights.")

    return w_val / w_val.sum()


def solve_qp(objective, constraints, w, max_w=None, verbose=False):
    """
    Robust CVXPY solver wrapper.
    Tries CLARABEL first, then OSQP, then SCS.
    """
    prob = cp.Problem(objective, constraints)

    solvers = []

    installed = cp.installed_solvers()

    if "CLARABEL" in installed:
        solvers.append(cp.CLARABEL)

    if "OSQP" in installed:
        solvers.append(cp.OSQP)

    if "SCS" in installed:
        solvers.append(cp.SCS)

    last_error = None

    for solver in solvers:
        try:
            prob.solve(solver=solver, verbose=verbose)

            if prob.status in ["optimal", "optimal_inaccurate"] and w.value is not None:
                return clean_weights(w.value, max_w=max_w)

        except Exception as e:
            last_error = e
            continue

    raise RuntimeError(
        f"All solvers failed. Problem status: {prob.status}. Last error: {last_error}"
    )

In [ ]:
def min_variance(mu, Sigma, max_w=0.20):
    n = len(mu)

    if n * max_w < 1:
        raise ValueError(
            f"Infeasible max_w={max_w:.2%} for {n} assets. "
            f"Need n * max_w >= 1."
        )

    Sigma = clean_cov(Sigma)

    w = cp.Variable(n)

    cons = [
        cp.sum(w) == 1,
        w >= 0,
        w <= max_w,
    ]

    obj = cp.Minimize(
        cp.quad_form(w, cp.psd_wrap(Sigma))
    )

    return solve_qp(obj, cons, w, max_w=max_w)


def mean_variance(mu, Sigma, risk_aversion=5.0, max_w=0.20):
    mu = np.asarray(mu, dtype=float).reshape(-1)
    Sigma = clean_cov(Sigma)

    n = len(mu)

    if n * max_w < 1:
        raise ValueError(
            f"Infeasible max_w={max_w:.2%} for {n} assets. "
            f"Need n * max_w >= 1."
        )

    w = cp.Variable(n)

    cons = [
        cp.sum(w) == 1,
        w >= 0,
        w <= max_w,
    ]

    obj = cp.Maximize(
        mu @ w
        - 0.5 * risk_aversion * cp.quad_form(w, cp.psd_wrap(Sigma))
    )

    return solve_qp(obj, cons, w, max_w=max_w)


def max_sharpe(mu, Sigma, rf=RF_ANNUAL, max_w=0.20, n_grid=60):
    """
    Trace the efficient frontier over a grid of risk aversions;
    pick the max Sharpe portfolio.
    """
    Sigma = clean_cov(Sigma)

    best = None
    best_s = -np.inf

    for ra in np.geomspace(0.5, 200, n_grid):
        try:
            w = mean_variance(mu, Sigma, risk_aversion=ra, max_w=max_w)

            vol = np.sqrt(w @ Sigma @ w)
            if vol <= 0:
                continue

            s = (mu @ w - rf) / vol

            if s > best_s:
                best = w
                best_s = s

        except Exception:
            continue

    if best is None:
        raise RuntimeError("max_sharpe failed for all risk-aversion values.")

    return best

In [ ]:
def cvar_optimization(returns, alpha=0.95, max_w=0.20, target_return=None, periods=12):
    """
    Rockafellar-Uryasev LP formulation:
    minimize CVaR of historical scenarios.
    """
    R = returns.values
    T, n = R.shape

    if n * max_w < 1:
        raise ValueError(
            f"Infeasible max_w={max_w:.2%} for {n} assets. "
            f"Need n * max_w >= 1."
        )

    w = cp.Variable(n)
    z = cp.Variable()
    u = cp.Variable(T)

    cvar = z + cp.sum(u) / (T * (1 - alpha))

    cons = [
        u >= 0,
        u >= -R @ w - z,
        cp.sum(w) == 1,
        w >= 0,
        w <= max_w,
    ]

    if target_return is not None:
        cons.append(returns.mean().values @ w * periods >= target_return)

    return solve_qp(cp.Minimize(cvar), cons, w, max_w=max_w)


In [ ]:
def robust_mv(mu, Sigma, mu_se, kappa=1.5, risk_aversion=5.0, max_w=0.20):
    """
    Robust mean-variance under box uncertainty on mu.
    """
    mu = np.asarray(mu, dtype=float).reshape(-1)
    mu_se = np.asarray(mu_se, dtype=float).reshape(-1)
    Sigma = clean_cov(Sigma)

    n = len(mu)

    if n * max_w < 1:
        raise ValueError(
            f"Infeasible max_w={max_w:.2%} for {n} assets. "
            f"Need n * max_w >= 1."
        )

    w = cp.Variable(n)

    worst_mu = mu @ w - kappa * (mu_se @ cp.abs(w))

    cons = [
        cp.sum(w) == 1,
        w >= 0,
        w <= max_w,
    ]

    obj = cp.Maximize(
        worst_mu
        - 0.5 * risk_aversion * cp.quad_form(w, cp.psd_wrap(Sigma))
    )

    return solve_qp(obj, cons, w, max_w=max_w)

In [ ]:
def black_litterman(returns, market_caps, P=None, Q=None, view_conf=None,
                    tau=0.05, delta=2.5, periods=12):
    """
    Black-Litterman: start from equilibrium returns implied by market-cap weights,
    then tilt by views. With no views, posterior = prior.
    """
    Sigma = returns.cov().values * periods
    Sigma = clean_cov(Sigma)

    market_caps = np.asarray(market_caps, dtype=float)
    market_caps = np.nan_to_num(market_caps, nan=0.0, posinf=0.0, neginf=0.0)

    if market_caps.sum() <= 0:
        raise ValueError("market_caps must contain at least one positive value.")

    w_mkt = market_caps / market_caps.sum()

    pi = delta * Sigma @ w_mkt

    if P is None:
        return pi, Sigma

    P = np.atleast_2d(P)
    Q = np.atleast_1d(Q)

    if view_conf is None:
        view_conf = np.ones(len(Q))

    view_conf = np.asarray(view_conf, dtype=float)

    Omega = np.diag(
        np.diag(P @ (tau * Sigma) @ P.T) / view_conf
    )

    inv_tau_sigma = np.linalg.pinv(tau * Sigma)
    inv_omega = np.linalg.pinv(Omega)

    M = np.linalg.pinv(
        inv_tau_sigma + P.T @ inv_omega @ P
    )

    mu_bl = M @ (
        inv_tau_sigma @ pi
        + P.T @ inv_omega @ Q
    )

    Sigma_bl = Sigma + M
    Sigma_bl = clean_cov(Sigma_bl)

    return mu_bl, Sigma_bl

In [ ]:
# mu / Sigma used by the static (in-sample) analyses in Sections 3 and 7.
# "In-sample" is the point -- this is a *snapshot* fit on all data through today,
# the same way you'd estimate inputs to build a portfolio right now. It is NOT
# what the walk-forward backtest in Section 8 uses (that re-estimates inside
# each rolling window, which is the only out-of-sample-honest version).
mu_s, _ = mu_sigma(rets_m_full)
Sig_lw = lw_sigma(rets_m_full)

# Baseline portfolio weights, used to annotate the efficient frontier plot and
# as one row of the Section-7 risk-model comparison.
W = pd.DataFrame({
    "MinVar":     min_variance(mu_s, Sig_lw),
    "MaxSharpe":  max_sharpe(mu_s, Sig_lw),
    "RiskParity": risk_parity(Sig_lw),
}, index=TICKERS)
W.round(3)


### Efficient frontier, drawdown and downside risk

The frontier below uses Ledoit–Wolf covariance and the 20% position cap. Individual assets sit *below* the frontier (diversification benefit). Note how compressed the frontier is — with 18 highly correlated growth names, the optimizer has limited raw material to diversify with.

In [ ]:
# Efficient frontier
ras = np.geomspace(0.5, 300, 40)
frontier = []
for ra in ras:
    w = mean_variance(mu_s, Sig_lw, ra)
    frontier.append((np.sqrt(w @ Sig_lw @ w), mu_s @ w))
frontier = np.array(frontier)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(frontier[:, 0], frontier[:, 1], "b-", lw=2, label="Efficient frontier (LW, capped)")
asset_vol = np.sqrt(np.diag(Sig_lw))
ax.scatter(asset_vol, mu_s, c="gray", s=25, label="Individual assets")
for i, t in enumerate(TICKERS):
    ax.annotate(t, (asset_vol[i], mu_s[i]), fontsize=7, alpha=0.7)
for name, marker, color in [("MinVar","s","green"), ("MaxSharpe","*","red"), ("RiskParity","D","orange")]:
    w = W[name].values
    ax.scatter(np.sqrt(w @ Sig_lw @ w), mu_s @ w, marker=marker, s=160, c=color, label=name, zorder=5)
ax.set_xlabel("Annualized volatility"); ax.set_ylabel("Annualized return (historical — inflated by selection bias)")
ax.set_title("Efficient frontier"); ax.legend(); plt.show()

# Drawdown comparison: in-sample MinVar vs equal weight vs SPY
ew = rets_d_full.mean(axis=1)
mv_r = rets_d_full @ W["MinVar"].values
fig, ax = plt.subplots()
for r, lbl in [(mv_r, "MinVar"), (ew, "Equal weight"), (bench_d["SPY"].loc[rets_d_full.index], "SPY")]:
    drawdown_series(r).plot(ax=ax, label=lbl)
ax.set_title("Drawdowns (in-sample weights — illustrative only)"); ax.legend(); plt.show()


## 4. Factor investing

**Why factors?** With n=18 assets, the sample covariance has 171 free parameters estimated from limited data. A k-factor model needs only ~18k + 18 parameters — a massive noise reduction. We extract **statistical factors** via PCA, then later compare against **nonlinear** autoencoder factors.

**Fama–French context.** Academic factors (Market, Size, Value, Profitability, Investment, Momentum) are *economically interpretable* but built from thousands of stocks. Our 18 mega-caps are nearly all large-growth-quality, so FF loadings would be degenerate (everything loads on Market + negative Value). Statistical factors are the right tool at this universe size. **Macro factors** (rates, inflation, USD, oil) could be added as exogenous regressors — we leave placeholders since FRED access varies by environment.

In [ ]:
# PCA on standardized daily returns (full panel). Standardizing = factors from the
# CORRELATION matrix, so high-vol names (TSLA, NVDA) don't mechanically dominate PC1.
scaler = StandardScaler()
X = scaler.fit_transform(rets_d_full.values)

pca = PCA().fit(X)
evr = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(range(1, len(evr)+1), evr)
axes[0].plot(range(1, len(evr)+1), np.cumsum(evr), "ro-", ms=4)
axes[0].set_title(f"PCA explained variance — PC1 alone: {evr[0]:.0%} (the 'market/growth' factor)")
axes[0].set_xlabel("Component")

loadings = pd.DataFrame(pca.components_[:3].T, index=TICKERS, columns=["PC1", "PC2", "PC3"])
loadings.plot.bar(ax=axes[1], width=0.85)
axes[1].set_title("Loadings: PC1 ≈ market beta; PC2/PC3 separate defensives & themes")
plt.tight_layout(); plt.show()

K_FACTORS = int(np.searchsorted(np.cumsum(evr), 0.80) + 1)
print(f"Factors needed for 80% of variance: {K_FACTORS} → we use k={K_FACTORS} for PCA and AE latent dims")

def pca_factor_cov(returns, k=K_FACTORS, periods=12):
    """Factor-model covariance: B F B' + D (systematic + diagonal idiosyncratic).
    Computed on raw (unstandardized) returns so the output is a true return covariance."""
    R = returns.values - returns.values.mean(0)
    p = PCA(n_components=k).fit(R)
    B = p.components_.T * np.sqrt(p.explained_variance_)        # n x k loadings
    F = R @ p.components_.T                                     # factor scores
    resid = R - F @ p.components_
    return (B @ B.T + np.diag(resid.var(0))) * periods


## 5. Autoencoder modeling

**Setup.** Each training sample is one day's cross-section of 18 standardized returns. The encoder compresses 18 → k latent factors; the decoder reconstructs. PCA is exactly a **linear** autoencoder with MSE loss — so the AE can only add value through nonlinearity. With 18 assets, the honest prior is "marginal gains at best." We test it rather than assume it.

Three architectures:
1. **Basic AE** — nonlinear factor discovery
2. **Denoising AE** — trained to reconstruct clean returns from noise-corrupted inputs; the reconstructions give us a *denoised covariance matrix*
3. **Variational AE** — probabilistic latent space; lets us sample synthetic return scenarios and inspect latent market regimes

**Anti-overfitting discipline:** chronological train/validation split (no shuffling across the split boundary), early stopping on validation loss, small networks, weight decay.

In [ ]:
class AE(nn.Module):
    def __init__(self, n_assets, k, hidden=32):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_assets, hidden), nn.SiLU(),
                                     nn.Linear(hidden, k))
        self.decoder = nn.Sequential(nn.Linear(k, hidden), nn.SiLU(),
                                     nn.Linear(hidden, n_assets))
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

class VAE(nn.Module):
    def __init__(self, n_assets, k, hidden=32):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(n_assets, hidden), nn.SiLU())
        self.mu_head, self.lv_head = nn.Linear(hidden, k), nn.Linear(hidden, k)
        self.decoder = nn.Sequential(nn.Linear(k, hidden), nn.SiLU(),
                                     nn.Linear(hidden, n_assets))
    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.mu_head(h), self.lv_head(h)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decoder(z), mu, logvar

def train_model(model, X_tr, X_va, epochs=400, lr=1e-3, batch=64,
                noise_std=0.0, beta_kl=0.0, patience=40):
    """Shared training loop. noise_std>0 → denoising AE. beta_kl>0 → VAE."""
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    tr = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32)),
                    batch_size=batch, shuffle=True)
    Xv = torch.tensor(X_va, dtype=torch.float32, device=DEVICE)
    best, best_state, since = np.inf, None, 0
    hist = {"train": [], "val": []}
    for ep in range(epochs):
        model.train(); tl = 0.0
        for (xb,) in tr:
            xb = xb.to(DEVICE)
            inp = xb + noise_std * torch.randn_like(xb) if noise_std > 0 else xb
            opt.zero_grad()
            if beta_kl > 0:
                rec, mu, logvar = model(inp)
                kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                loss = nn.functional.mse_loss(rec, xb) + beta_kl * kl
            else:
                rec, _ = model(inp)
                loss = nn.functional.mse_loss(rec, xb)
            loss.backward(); opt.step(); tl += loss.item() * len(xb)
        model.eval()
        with torch.no_grad():
            out = model(Xv)
            vl = nn.functional.mse_loss(out[0], Xv).item()
        hist["train"].append(tl / len(X_tr)); hist["val"].append(vl)
        if vl < best - 1e-6:
            best, best_state, since = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            since += 1
            if since >= patience: break
    model.load_state_dict(best_state)
    return model, hist

# Chronological split: first 80% train, last 20% validation. NEVER shuffle time series
# across the split — that leaks regime information and flatters validation loss.
split = int(len(X) * 0.8)
X_tr, X_va = X[:split], X[split:]
N = X.shape[1]

ae,  h_ae  = train_model(AE(N, K_FACTORS),  X_tr, X_va)
dae, h_dae = train_model(AE(N, K_FACTORS),  X_tr, X_va, noise_std=0.5)
vae, h_vae = train_model(VAE(N, K_FACTORS), X_tr, X_va, beta_kl=2e-3)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=False)
for ax_, h, t in zip(axes, [h_ae, h_dae, h_vae], ["Basic AE", "Denoising AE", "VAE"]):
    ax_.plot(h["train"], label="train"); ax_.plot(h["val"], label="val")
    ax_.set_title(t); ax_.legend(fontsize=8)
plt.suptitle("Training curves — watch for a widening train/val gap (overfitting)"); plt.tight_layout(); plt.show()


### Reconstruction quality: AE vs PCA — the honest horse race

Same latent dimension `k` for both. Evaluated **only on the validation period** (out-of-sample). If PCA wins or ties, that is a legitimate finding: 18 large-caps are dominated by one linear market factor, leaving little nonlinear structure to exploit.

In [ ]:
pca_k = PCA(n_components=K_FACTORS).fit(X_tr)
pca_rec_va = pca_k.inverse_transform(pca_k.transform(X_va))
pca_mse = np.mean((X_va - pca_rec_va) ** 2)

with torch.no_grad():
    Xv = torch.tensor(X_va, dtype=torch.float32, device=DEVICE)
    ae_mse  = nn.functional.mse_loss(ae(Xv)[0],  Xv).item()
    dae_mse = nn.functional.mse_loss(dae(Xv)[0], Xv).item()

comp = pd.Series({"PCA": pca_mse, "Basic AE": ae_mse, "Denoising AE": dae_mse},
                 name=f"Validation reconstruction MSE (k={K_FACTORS})")
print(comp.round(4).to_string())
winner = comp.idxmin()
print(f"\nLowest validation error: {winner}. "
      "With only 18 assets, expect this race to be close — the value of the AE here is "
      "less in reconstruction and more in the latent structure (clustering, anomalies, regimes).")


### Latent embeddings, asset clustering, and hidden concentration

We embed each **asset** in latent space via its correlation with each latent factor (the AE analogue of factor loadings), then cluster with KMeans. **This is where thematic labels get exposed:** if NVDA, AMD, AVGO, MSFT and META all land in one cluster, your "five themes" are really one trade.

In [ ]:
with torch.no_grad():
    Z = ae.encoder(torch.tensor(X, dtype=torch.float32, device=DEVICE)).cpu().numpy()
Z_df = pd.DataFrame(Z, index=rets_d_full.index, columns=[f"L{i+1}" for i in range(K_FACTORS)])

# Asset loadings on latent factors: corr(asset return, latent factor)
asset_emb = pd.DataFrame(
    [[np.corrcoef(rets_d_full[t], Z_df[f])[0, 1] for f in Z_df] for t in TICKERS],
    index=TICKERS, columns=Z_df.columns)

n_clusters = 5
km = KMeans(n_clusters=n_clusters, n_init=20, random_state=SEED).fit(asset_emb.values)
clusters = pd.Series(km.labels_, index=TICKERS, name="cluster")

# 2-D view of asset embeddings
emb2 = PCA(n_components=2).fit_transform(asset_emb.values)
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(emb2[:, 0], emb2[:, 1], c=km.labels_, cmap="tab10", s=120)
for i, t in enumerate(TICKERS):
    ax.annotate(f"{t}\n({UNIVERSE[t]})", (emb2[i, 0], emb2[i, 1]), fontsize=7, ha="center", va="bottom")
ax.set_title("Assets in autoencoder latent space — clusters reveal TRUE diversification structure")
plt.show()

print("Latent-space clusters vs. claimed themes:")
for c in range(n_clusters):
    members = clusters[clusters == c].index.tolist()
    print(f"  Cluster {c}: {members}")
print("\nIf one cluster contains most of the tech complex, position caps per CLUSTER "
      "(not per ticker) are the right risk control.")


### Latent activation matrix and regime interpretation

The autoencoder should not be treated as a direct weight generator. Its more defensible role is to create a **latent activation matrix**: one row per date, one column per latent factor. This lets us ask whether the hidden factors behave like market regimes, volatility states, momentum states, or reconstruction-error stress states.

This section makes the neural network less of a black box by linking each latent activation to observable financial quantities.

In [ ]:
# 1) Latent activation matrix: date x hidden AE factor
latent_activation_matrix = Z_df.copy()
latent_z = (latent_activation_matrix - latent_activation_matrix.mean()) / latent_activation_matrix.std(ddof=0)
latent_z = latent_z.replace([np.inf, -np.inf], np.nan).fillna(0.0)


def safe_corr(a, b, min_obs=30):
    tmp = pd.concat([pd.Series(a), pd.Series(b)], axis=1).dropna()
    if len(tmp) < min_obs:
        return np.nan
    return float(tmp.iloc[:, 0].corr(tmp.iloc[:, 1]))


ew_aligned = rets_d_full.mean(axis=1).reindex(latent_z.index)
spy_aligned = bench_d['SPY'].reindex(latent_z.index)
vol21 = ew_aligned.rolling(21).std() * np.sqrt(TRADING_DAYS)
mom63 = ew_aligned.rolling(63).mean() * TRADING_DAYS
rec_aligned = rec_err.reindex(latent_z.index) if 'rec_err' in globals() else pd.Series(np.nan, index=latent_z.index)

latent_activation_diagnostics = pd.DataFrame(index=latent_z.columns)
latent_activation_diagnostics['Corr_EW_daily_return'] = [safe_corr(latent_z[c], ew_aligned) for c in latent_z.columns]
latent_activation_diagnostics['Corr_SPY_daily_return'] = [safe_corr(latent_z[c], spy_aligned) for c in latent_z.columns]
latent_activation_diagnostics['Corr_21d_realized_vol'] = [safe_corr(latent_z[c], vol21) for c in latent_z.columns]
latent_activation_diagnostics['Corr_63d_momentum'] = [safe_corr(latent_z[c], mom63) for c in latent_z.columns]
latent_activation_diagnostics['Corr_DAE_recon_error'] = [safe_corr(latent_z[c], rec_aligned) for c in latent_z.columns]

print('Latent activation matrix shape:', latent_activation_matrix.shape)
print('Interpretation: high absolute correlations suggest what each hidden factor is tracking.')
display(latent_activation_diagnostics.round(3))

# Heatmap of smoothed latent states through time.
latent_smooth = latent_z.rolling(21, min_periods=5).mean().dropna()
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(latent_smooth.T, aspect='auto', interpolation='nearest')
ax.set_yticks(range(latent_smooth.shape[1]))
ax.set_yticklabels(latent_smooth.columns)
xticks = np.linspace(0, len(latent_smooth) - 1, min(8, len(latent_smooth))).astype(int)
ax.set_xticks(xticks)
ax.set_xticklabels([latent_smooth.index[i].strftime('%Y-%m') for i in xticks], rotation=45, ha='right')
ax.set_title('21-day smoothed AE latent activation matrix')
plt.colorbar(im, ax=ax, label='latent z-score')
plt.tight_layout()
plt.show()

# Regime snapshot: average latent activation on normal vs stress days.
if 'stress' in globals():
    latent_regime_snapshot = pd.DataFrame({
        'Normal_days_mean_z': latent_z.loc[stress.reindex(latent_z.index).fillna(0).astype(bool) == False].mean(),
        'AE_stress_days_mean_z': latent_z.loc[stress.reindex(latent_z.index).fillna(0).astype(bool)].mean(),
        'Difference_stress_minus_normal': latent_z.loc[stress.reindex(latent_z.index).fillna(0).astype(bool)].mean() - latent_z.loc[stress.reindex(latent_z.index).fillna(0).astype(bool) == False].mean(),
    })
    print('Latent activation by AE stress flag:')
    display(latent_regime_snapshot.round(3))

### Anomaly detection & regime shifts

Daily reconstruction error from the denoising AE = "how unusual was today's cross-section of returns relative to learned structure?" Spikes flag stress regimes (correlations jump toward 1, the factor structure breaks). We define a simple **stress regime** as error above its rolling 95th percentile — used later for regime-switching allocation.

In [ ]:
with torch.no_grad():
    Xall = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    rec_err = ((dae(Xall)[0] - Xall) ** 2).mean(dim=1).cpu().numpy()
rec_err = pd.Series(rec_err, index=rets_d_full.index, name="recon_error")

thresh = rec_err.rolling(252, min_periods=126).quantile(0.95)
stress = (rec_err > thresh).astype(int)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(rec_err, lw=0.6); axes[0].plot(thresh, "r--", lw=1, label="rolling 95th pct")
axes[0].set_title("DAE reconstruction error — spikes = cross-sectional structure breaking down")
axes[0].legend()
cum_spy = (1 + bench_d["SPY"].loc[rets_d_full.index]).cumprod()
axes[1].plot(cum_spy, "k", lw=1)
axes[1].fill_between(rec_err.index, 0, 1, where=stress.astype(bool),
                     transform=axes[1].get_xaxis_transform(), color="red", alpha=0.15)
axes[1].set_title("SPY with AE-flagged stress regimes shaded")
plt.tight_layout(); plt.show()
print(f"Share of days flagged as stress: {stress.mean():.1%}")
print("Check: do shaded bands align with COVID crash (Mar 2020) and the 2022 drawdown?")


### VAE: synthetic returns and latent market regimes

The VAE's probabilistic latent space supports two things a plain AE can't:
1. **Synthetic scenario generation** — sample z ~ N(0, I), decode, de-standardize → simulated daily return cross-sections that respect the learned (possibly nonlinear) dependence structure. Useful for stress testing beyond the single observed history.
2. **Regime inspection** — coloring the latent means by realized market volatility shows whether calm and turbulent days occupy different regions.

**Caveat:** VAEs notoriously under-generate tails (Gaussian prior + MSE likelihood). The QQ-style comparison below makes this visible instead of hiding it — for crash risk, EVT (Section 6) is the more trustworthy tool.

In [ ]:
with torch.no_grad():
    _, mu_z, _ = vae(torch.tensor(X, dtype=torch.float32, device=DEVICE))
    mu_z = mu_z.cpu().numpy()
    z_samp = torch.randn(5000, K_FACTORS, device=DEVICE)
    synth = vae.decoder(z_samp).cpu().numpy()
synth_rets = synth * scaler.scale_ + scaler.mean_     # back to return units

mkt_vol = rets_d_full.mean(axis=1).rolling(21).std() * np.sqrt(252)
emb2t = PCA(n_components=2).fit_transform(mu_z)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc = axes[0].scatter(emb2t[:, 0], emb2t[:, 1], c=mkt_vol.values, cmap="inferno", s=4)
plt.colorbar(sc, ax=axes[0], label="21d realized vol (ann.)")
axes[0].set_title("VAE latent means colored by market vol — regimes separate visibly")

real_port = rets_d_full.mean(axis=1)
synth_port = synth_rets.mean(axis=1)
qs = np.linspace(0.001, 0.999, 200)
axes[1].plot(np.quantile(real_port, qs), np.quantile(synth_port, qs), "b.", ms=3)
lims = [real_port.min(), real_port.max()]
axes[1].plot(lims, lims, "r--", lw=1)
axes[1].set_xlabel("Real EW-portfolio return quantiles"); axes[1].set_ylabel("VAE synthetic quantiles")
axes[1].set_title("VAE vs reality: note the thin synthetic tails — a known VAE failure mode")
plt.tight_layout(); plt.show()


### Autoencoder factors → covariance matrices for optimization

Two AE-derived risk models, mirroring the PCA factor covariance:
- **AE-factor covariance:** regress asset returns on latent factors → `B Σ_z B' + D`
- **DAE-denoised covariance:** covariance of the DAE's reconstructed returns plus residual diagonal — the reconstruction acts as a structural noise filter, conceptually similar to eigenvalue clipping

In [ ]:
def ae_factor_cov(returns_d, Z_factors, periods=TRADING_DAYS):
    """Sigma = B Cov(z) B' + diag(residual var), estimated by OLS of returns on latent factors."""
    R = returns_d.values - returns_d.values.mean(0)
    Zc = Z_factors - Z_factors.mean(0)
    B, *_ = np.linalg.lstsq(Zc, R, rcond=None)          # k x n
    resid = R - Zc @ B
    Sigma_z = np.cov(Zc, rowvar=False)
    return (B.T @ Sigma_z @ B + np.diag(resid.var(0))) * periods

def dae_denoised_cov(returns_d, dae_model, scaler, periods=TRADING_DAYS, resid_frac=1.0):
    Xs = scaler.transform(returns_d.values)
    with torch.no_grad():
        rec = dae_model(torch.tensor(Xs, dtype=torch.float32, device=DEVICE))[0].cpu().numpy()
    rec_rets = rec * scaler.scale_ + scaler.mean_
    resid = returns_d.values - rec_rets
    return (np.cov(rec_rets, rowvar=False) + resid_frac * np.diag(resid.var(0))) * periods

Sig_ae  = ae_factor_cov(rets_d_full, Z)
Sig_dae = dae_denoised_cov(rets_d_full, dae, scaler)
print("Covariance condition numbers (lower = more stable to invert/optimize):")
for name, S in [("Sample (daily, ann.)", rets_d_full.cov().values * 252),
                ("Ledoit-Wolf", lw_sigma(rets_d_full, 252)),
                ("PCA factor", pca_factor_cov(rets_d_full, periods=252)),
                ("AE factor", Sig_ae), ("DAE denoised", Sig_dae)]:
    print(f"  {name:22s}: {np.linalg.cond(S):,.0f}")


### Random Matrix Theory covariance denoising

This is the cleanest way to make the physics idea concrete. The empirical correlation matrix contains signal and noise. Random Matrix Theory gives a benchmark for the eigenvalues expected from a random correlation matrix. Eigenvalues inside the noise band are clipped/averaged; large eigenvalues are treated as market/sector modes.

The result is a denoised covariance matrix that can be fed into min-variance optimization and Black-Litterman.

In [ ]:
# 2) Random Matrix Theory covariance denoising

def rmt_denoised_cov(returns, periods=TRADING_DAYS, return_info=False):
    """
    RMT eigenvalue clipping for a return covariance matrix.

    Input returns can be daily or monthly. Set periods=252 for daily annualization
    and periods=12 for monthly annualization.
    """
    R = pd.DataFrame(returns).dropna(how='any')
    if R.shape[0] < R.shape[1] + 5:
        # Small-sample fallback: shrinkage is safer than pretending RMT is precise.
        Sigma = lw_sigma(R, periods=periods)
        info = pd.DataFrame({
            'Eigenvalue_raw': np.linalg.eigvalsh(R.corr().values),
            'Eigenvalue_clean': np.linalg.eigvalsh(R.corr().values),
            'Is_signal_mode': False,
            'Lambda_plus': np.nan,
            'Q_T_over_N': R.shape[0] / R.shape[1],
        })
        return (Sigma, info) if return_info else Sigma

    Xs = (R - R.mean()) / R.std(ddof=0)
    Xs = Xs.replace([np.inf, -np.inf], np.nan).dropna(how='any')
    T, N = Xs.shape
    q = T / N

    corr_raw = np.corrcoef(Xs.values, rowvar=False)
    corr_raw = np.nan_to_num(corr_raw, nan=0.0, posinf=0.0, neginf=0.0)
    corr_raw = 0.5 * (corr_raw + corr_raw.T)
    np.fill_diagonal(corr_raw, 1.0)

    evals, evecs = np.linalg.eigh(corr_raw)
    order = np.argsort(evals)
    evals, evecs = evals[order], evecs[:, order]

    lambda_minus = (1.0 - np.sqrt(1.0 / q)) ** 2
    lambda_plus = (1.0 + np.sqrt(1.0 / q)) ** 2
    signal_mask = evals > lambda_plus
    noise_mask = ~signal_mask

    evals_clean = evals.copy()
    if noise_mask.any():
        evals_clean[noise_mask] = evals[noise_mask].mean()

    corr_clean = evecs @ np.diag(evals_clean) @ evecs.T
    corr_clean = 0.5 * (corr_clean + corr_clean.T)
    d = np.sqrt(np.clip(np.diag(corr_clean), 1e-12, None))
    corr_clean = corr_clean / np.outer(d, d)
    corr_clean = np.clip(corr_clean, -0.999, 0.999)
    np.fill_diagonal(corr_clean, 1.0)

    vols = R.std(ddof=1).values * np.sqrt(periods)
    Sigma = corr_clean * np.outer(vols, vols)
    Sigma = clean_cov(Sigma)

    info = pd.DataFrame({
        'Eigenvalue_raw': evals,
        'Eigenvalue_clean': evals_clean,
        'Is_signal_mode': signal_mask,
        'Lambda_minus': lambda_minus,
        'Lambda_plus': lambda_plus,
        'Q_T_over_N': q,
    })
    return (Sigma, info) if return_info else Sigma


Sig_rmt, rmt_eigen_info = rmt_denoised_cov(rets_m_full, periods=12, return_info=True)
Sig_rmt_daily = rmt_denoised_cov(rets_d_full, periods=TRADING_DAYS)

rmt_summary = pd.Series({
    'T_observations_monthly': len(rets_m_full.dropna()),
    'N_assets': len(TICKERS),
    'Q_T_over_N': rmt_eigen_info['Q_T_over_N'].iloc[0],
    'Lambda_plus_noise_edge': rmt_eigen_info['Lambda_plus'].iloc[0],
    'Signal_modes_above_noise': int(rmt_eigen_info['Is_signal_mode'].sum()),
    'Noise_eigenvalue_share': 1.0 - float(rmt_eigen_info['Is_signal_mode'].mean()),
})
print('RMT denoising summary')
display(rmt_summary.round(3).to_frame('Value'))

print('Largest raw eigenvalues: market/sector modes that survive the RMT noise filter')
display(rmt_eigen_info.sort_values('Eigenvalue_raw', ascending=False).head(8).round(3))

rmt_condition_table = pd.DataFrame({
    'ConditionNumber': {
        'Sample monthly covariance': np.linalg.cond(clean_cov(rets_m_full.cov().values * 12)),
        'Ledoit-Wolf monthly covariance': np.linalg.cond(clean_cov(Sig_lw)),
        'RMT-denoised monthly covariance': np.linalg.cond(clean_cov(Sig_rmt)),
        'AE-factor daily covariance': np.linalg.cond(clean_cov(Sig_ae)),
        'DAE-denoised daily covariance': np.linalg.cond(clean_cov(Sig_dae)),
    }
})
print('Lower condition numbers usually mean more stable optimization inputs.')
display(rmt_condition_table.round(1))

w_rmt_static = min_variance(mu_s, Sig_rmt)
print('Static RMT min-variance concentration:')
print(f'HHI={herfindahl(w_rmt_static):.3f}, top weight={pd.Series(w_rmt_static, index=TICKERS).max():.1%}')

### Correlation network and hidden concentration risk

A portfolio can look diversified by ticker count while still being one connected trade. Here, each stock is a node and each strong absolute correlation is an edge. The network view identifies central stocks and latent crowding that simple sector labels miss.

In [ ]:
# 3) Correlation network analysis without relying on extra graph libraries
from scipy.sparse.csgraph import minimum_spanning_tree


def correlation_network_diagnostics(returns, threshold=0.55):
    corr_net = pd.DataFrame(returns).corr().reindex(index=TICKERS, columns=TICKERS)
    abs_corr = corr_net.abs()
    adj = (abs_corr >= threshold).astype(float)
    np.fill_diagonal(adj.values, 0.0)

    weighted_edges = abs_corr.where(adj.astype(bool), 0.0)
    weighted_degree = weighted_edges.sum(axis=1)

    A = weighted_edges.values
    if np.allclose(A, 0):
        eig_cent = np.ones(len(TICKERS)) / len(TICKERS)
    else:
        vals, vecs = np.linalg.eigh(A)
        eig_cent = np.abs(vecs[:, -1])
        eig_cent = eig_cent / eig_cent.sum() if eig_cent.sum() > 0 else np.ones(len(TICKERS)) / len(TICKERS)

    # Simple unweighted clustering coefficient: how connected are my neighbors?
    adj_arr = adj.values.astype(int)
    clustering = []
    for i in range(len(TICKERS)):
        neigh = np.where(adj_arr[i] > 0)[0]
        k = len(neigh)
        if k < 2:
            clustering.append(0.0)
        else:
            actual = adj_arr[np.ix_(neigh, neigh)].sum() / 2
            possible = k * (k - 1) / 2
            clustering.append(actual / possible)

    diag = pd.DataFrame({
        'WeightedDegree': weighted_degree,
        'EigenvectorCentrality': eig_cent,
        'ClusteringCoeff': clustering,
        'Theme': [UNIVERSE[t] for t in TICKERS],
        'LatentCluster': clusters.reindex(TICKERS).values,
    }, index=TICKERS).sort_values('EigenvectorCentrality', ascending=False)

    # Minimum spanning tree from correlation distance.
    dist = np.sqrt(np.clip(2 * (1 - corr_net.clip(-1, 1).values), 0, None))
    mst = minimum_spanning_tree(dist).toarray()
    edge_rows = []
    for i, j in zip(*np.nonzero(mst)):
        edge_rows.append({
            'NodeA': TICKERS[i],
            'NodeB': TICKERS[j],
            'Correlation': corr_net.iloc[i, j],
            'Distance': dist[i, j],
        })
    mst_edges = pd.DataFrame(edge_rows).sort_values('Distance')
    return corr_net, adj, diag, mst_edges


corr_net, corr_adj, network_diag, mst_edges = correlation_network_diagnostics(rets_d_full, threshold=0.55)
print('Most central stocks in the correlation network:')
display(network_diag.head(10).round(3))
print('Closest links in the minimum spanning tree:')
display(mst_edges.head(12).round(3))

# Visualize the network using the already-computed AE asset embedding as coordinates.
pos = pd.DataFrame(emb2, index=TICKERS, columns=['x', 'y']) if 'emb2' in globals() else pd.DataFrame(PCA(n_components=2).fit_transform(asset_emb.values), index=TICKERS, columns=['x', 'y'])
fig, ax = plt.subplots(figsize=(9, 7))
for i, a in enumerate(TICKERS):
    for j, b in enumerate(TICKERS):
        if j <= i:
            continue
        if corr_adj.loc[a, b] > 0:
            ax.plot([pos.loc[a, 'x'], pos.loc[b, 'x']], [pos.loc[a, 'y'], pos.loc[b, 'y']], alpha=0.25, linewidth=1)
node_size = 5000 * network_diag.reindex(TICKERS)['EigenvectorCentrality'].values
ax.scatter(pos['x'], pos['y'], s=node_size)
for t in TICKERS:
    ax.annotate(t, (pos.loc[t, 'x'], pos.loc[t, 'y']), ha='center', va='center', fontsize=8)
ax.set_title('Correlation network: node size = eigenvector centrality, edges = |corr| >= 0.55')
plt.tight_layout()
plt.show()


def network_concentration_score(weights, diag=network_diag):
    w = pd.Series(weights, index=TICKERS).reindex(diag.index).fillna(0.0)
    cent = diag['EigenvectorCentrality']
    return float((w * cent).sum() / cent.mean())

print('Network concentration scores. A value above 1 means the portfolio is tilted toward central names.')
network_conc_static = pd.Series({
    'Ledoit-Wolf': network_concentration_score(PORT_W['Ledoit-Wolf']) if 'PORT_W' in globals() and 'Ledoit-Wolf' in PORT_W else np.nan,
    'RMT denoised': network_concentration_score(w_rmt_static),
    'Equal weight': network_concentration_score(np.ones(len(TICKERS)) / len(TICKERS)),
})
display(network_conc_static.round(3).to_frame('NetworkConcentration'))

## 6. Advanced portfolio construction

A grab-bag of production techniques. Each gets a working implementation or an honest conceptual treatment — labeled which is which.

| Technique | Treatment here |
|---|---|
| Regime-switching allocation | **Implemented** (AE stress flag → de-risk) |
| Volatility targeting | **Implemented** |
| Fractional Kelly | **Implemented** |
| Scenario stress testing | **Implemented** (historical + VAE-synthetic) |
| Entropy pooling | **Conceptual + minimal demo** |
| Copula dependence | **Implemented** (Gaussian vs Student-t tail dependence) |
| Extreme value theory | **Implemented** (GPD peaks-over-threshold) |
| Liquidity-adjusted optimization | **Conceptual framework** (all 18 names are mega-cap liquid; the constraint binds at AUM ~$10M+, not €1,000) |

In [ ]:
# --- Volatility targeting: scale exposure so realized vol ≈ target ---
def vol_target_returns(port_rets, target=0.12, lookback=21, max_lev=1.0):
    """Daily port returns -> vol-targeted returns. max_lev=1 (no leverage; rest in cash at RF).
    Uses YESTERDAY's vol estimate (shifted) — no look-ahead."""
    realized = port_rets.rolling(lookback).std() * np.sqrt(252)
    lev = (target / realized).clip(upper=max_lev).shift(1).fillna(1.0)
    return lev * port_rets + (1 - lev) * (RF_ANNUAL / 252)

# --- Fractional Kelly ---
def kelly_fraction(mu_excess, sigma2, fraction=0.5):
    """f* = mu/sigma^2 for a single risky asset (log-utility). Full Kelly is famously
    intolerable (50%+ drawdowns); half-Kelly gives ~75% of growth at ~half the variance.
    We apply it to the PORTFOLIO as a risky asset vs cash, capped at 1 (no leverage)."""
    return min(fraction * mu_excess / sigma2, 1.0)

ew_d = rets_d_full.mean(axis=1)
f_half = kelly_fraction(ann_return(ew_d) - RF_ANNUAL, ann_vol(ew_d) ** 2)
print(f"Half-Kelly fraction for the EW portfolio vs cash: {f_half:.2f} "
      "(>=1 means even half-Kelly says 'fully invested' — driven by hindsight-inflated mu!)")

# --- Regime-switching allocation: de-risk to 40% exposure when AE flags stress ---
def regime_switch_returns(port_rets, stress_flag, risk_on=1.0, risk_off=0.4):
    expo = stress_flag.reindex(port_rets.index).ffill().fillna(0)
    expo = expo.replace({1: risk_off, 0: risk_on}).shift(1).fillna(risk_on)  # trade NEXT day
    return expo * port_rets + (1 - expo) * (RF_ANNUAL / 252)

strategies_demo = {
    "EW buy & hold": ew_d,
    "EW vol-target 12%": vol_target_returns(ew_d),
    "EW + AE regime switch": regime_switch_returns(ew_d, stress),
}
perf_table(strategies_demo)


In [ ]:
fig, ax = plt.subplots()
for n, r in strategies_demo.items():
    (1 + r).cumprod().plot(ax=ax, label=n)
ax.set_title("Overlay strategies on the equal-weight core (in-sample, illustrative)")
ax.set_yscale("log"); ax.legend(); plt.show()


### Entropy pooling (conceptual + minimal demo)

Meucci's entropy pooling generalizes Black–Litterman: start from prior scenario probabilities (uniform over history), impose views as *constraints on moments*, and find the posterior probabilities **closest to the prior in relative entropy (KL divergence)**. The result reweights history rather than assuming normality.

Minimal demo: impose the view "expected market return = 0% (flat decade ahead)" and watch which historical days get up/down-weighted.

In [ ]:
def entropy_pool(returns_vec, view_mean, bracket=(-5000, 5000)):
    """
    Posterior probs p minimizing KL(p||uniform) s.t. E_p[r] = view_mean.
    Exponential tilting: p_i ∝ exp(theta * r_i).
    """

    r = np.asarray(returns_vec, dtype=float)
    r = r[np.isfinite(r)]

    if len(r) == 0:
        raise ValueError("returns_vec contains no finite values.")

    r_min, r_max = r.min(), r.max()

    if not (r_min <= view_mean <= r_max):
        raise ValueError(
            f"view_mean={view_mean:.6g} is infeasible. "
            f"It must lie between min={r_min:.6g} and max={r_max:.6g}."
        )

    def probs(theta):
        z = theta * r
        z -= z.max()          # stable softmax
        w = np.exp(z)
        return w / w.sum()

    def mean_under(theta):
        return probs(theta) @ r

    lo, hi = bracket

    f_lo = mean_under(lo) - view_mean
    f_hi = mean_under(hi) - view_mean

    if np.isclose(f_lo, 0):
        theta = lo
    elif np.isclose(f_hi, 0):
        theta = hi
    else:
        theta = brentq(lambda t: mean_under(t) - view_mean, lo, hi)

    return probs(theta)
# --- Demo: impose "flat decade ahead" (E[r] = 0) on the EW-portfolio's daily
# returns and see which historical days the posterior leans on. ---
ew_d_demo = rets_d_full.mean(axis=1)
p_post = entropy_pool(ew_d_demo.values, view_mean=0.0)
n_eff = 1.0 / np.sum(p_post ** 2)        # effective number of scenarios used

tilt = pd.DataFrame({
    "return": ew_d_demo.values,
    "prior_w": 1.0 / len(ew_d_demo),
    "posterior_w": p_post,
}, index=ew_d_demo.index)
tilt["ratio"] = tilt["posterior_w"] / tilt["prior_w"]

print(f"Posterior mean: {p_post @ ew_d_demo.values:.5f} (target was 0.0)")
print(f"Effective scenarios: {n_eff:.0f} of {len(ew_d_demo)} "
      f"({n_eff/len(ew_d_demo):.1%}) -- how much the view concentrates probability mass.")
print("\nMost UP-weighted days (down days -> made more probable to pull mean to 0):")
print(tilt.sort_values("ratio", ascending=False).head(5).round(4).to_string())
print("\nMost DOWN-weighted days (up days -> made less probable):")
print(tilt.sort_values("ratio").head(5).round(4).to_string())


### Copula-based dependence

Linear correlation misses what kills portfolios: **assets crashing together**. We compare the Gaussian copula (zero tail dependence — assets become independent in extreme tails, a dangerously optimistic assumption) against the Student-t copula (positive tail dependence). The empirical check: count joint extreme days vs what each copula predicts.

In [ ]:
from scipy.stats import t as t_dist, norm

# Empirical -> uniform via ranks (the empirical copula), on two correlated tech names
pair = rets_d_full[["NVDA", "AMD"]].dropna()
U = pair.rank() / (len(pair) + 1)

# Fit t-copula df by maximizing pseudo-likelihood over a small grid (kept simple deliberately)
z_g = norm.ppf(U.values)
rho_g = np.corrcoef(z_g.T)[0, 1]

def t_copula_ll(df, U):
    x = t_dist.ppf(U.values, df)
    rho = np.corrcoef(x.T)[0, 1]
    R = np.array([[1, rho], [rho, 1]])
    # t-copula log-likelihood = joint mvt logpdf minus the marginal t logpdfs
    return (st.multivariate_t.logpdf(x, shape=R, df=df)
            - t_dist.logpdf(x[:, 0], df) - t_dist.logpdf(x[:, 1], df)).sum()

dfs = [3, 5, 8, 12, 20, 30]
lls = [t_copula_ll(d, U) for d in dfs]
df_hat = dfs[int(np.argmax(lls))]

# Lower tail dependence: lambda_L = 2 * t_{df+1}( -sqrt((df+1)(1-rho)/(1+rho)) ); Gaussian: 0
x_t = t_dist.ppf(U.values, df_hat); rho_t = np.corrcoef(x_t.T)[0, 1]
lam_L = 2 * t_dist.cdf(-np.sqrt((df_hat + 1) * (1 - rho_t) / (1 + rho_t)), df_hat + 1)

q = 0.05
joint_emp = ((U.iloc[:, 0] < q) & (U.iloc[:, 1] < q)).mean()
print(f"NVDA/AMD — Gaussian-copula rho: {rho_g:.2f} | fitted t-copula df: {df_hat}, rho: {rho_t:.2f}")
print(f"Lower tail dependence λ_L: t-copula = {lam_L:.2f}, Gaussian = 0.00")
print(f"Empirical P(both in worst 5% same day): {joint_emp:.3f}  vs independence: {q*q:.4f}")
print("→ Joint crashes are ~an order of magnitude more likely than independence implies. "
      "Risk models built on Gaussian dependence systematically understate this.")


### Extreme value theory: crash-risk analysis

Peaks-over-threshold: fit a Generalized Pareto Distribution to portfolio losses beyond the 95th percentile. The shape parameter ξ tells you the tail type — ξ > 0 means power-law (Pareto) tails, where the worst loss you've seen is *not* a good estimate of the worst loss possible.

In [ ]:
losses = -ew_d.dropna()
u = np.quantile(losses, 0.95)
exceed = losses[losses > u] - u
xi, loc, beta = st.genpareto.fit(exceed, floc=0)

def evt_var_cvar(alpha):
    n, nu = len(losses), len(exceed)
    var = u + beta/xi * (((n/nu) * (1 - alpha)) ** (-xi) - 1)
    cvar = (var + beta - xi * u) / (1 - xi)
    return var, cvar

print(f"GPD fit on EW-portfolio daily losses beyond {u:.2%}: xi = {xi:.3f}, beta = {beta:.4f}")
print(f"xi > 0 → heavy (Pareto-type) tail" if xi > 0 else "xi <= 0 → bounded/exponential tail")
for a in (0.99, 0.999):
    v, c = evt_var_cvar(a)
    emp_v = np.quantile(losses, a)
    print(f"  {a:.1%} daily VaR — EVT: {v:.2%} | empirical: {emp_v:.2%} | EVT CVaR: {c:.2%}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(exceed, bins=40, density=True, alpha=0.6, label="Loss exceedances")
xs = np.linspace(0, exceed.max(), 200)
ax.plot(xs, st.genpareto.pdf(xs, xi, 0, beta), "r-", lw=2, label=f"GPD fit (ξ={xi:.2f})")
ax.set_title("EVT: tail of daily losses, equal-weight portfolio"); ax.legend(); plt.show()


### Liquidity-adjusted optimization (framework)

Market impact cost ≈ `spread/2 + η·σ·√(Q/ADV)` (square-root law). A liquidity-aware optimizer subtracts expected impact from `mu` and caps positions at a small fraction of average daily volume.

**Why we don't run it:** at €1,000 across mega-caps trading $1–30B/day, participation is ~10⁻⁸ of ADV — impact is zero to machine precision. The framework matters from roughly $10M AUM in these names (much earlier in small-caps). Including a fake binding constraint would be theater; knowing *when* the constraint binds is the actual lesson.

In [ ]:
# Market-cap weights for the Black-Litterman equilibrium prior.
# (Needed by black_litterman() below -- pulled live since cap rankings drift;
# hardcoding them would silently go stale.)
mcaps = pd.Series(index=TICKERS, dtype=float)
for t in TICKERS:
    try:
        mcaps[t] = yf.Ticker(t).fast_info["market_cap"]
    except Exception:
        mcaps[t] = np.nan
if mcaps.isna().any():
    missing = mcaps[mcaps.isna()].index.tolist()
    print(f"Market cap fetch failed for {missing}; filling with cross-sectional median.")
    mcaps = mcaps.fillna(mcaps.median())
mcaps.sort_values(ascending=False).round(-9)


## 7. Portfolio optimization across risk models - the main comparison

Eight portfolios, identical optimizer settings where possible, differing **only in the risk/return model**:

1. Sample covariance
2. Ledoit-Wolf shrinkage
3. PCA-factor covariance
4. AE-factor covariance
5. DAE-denoised covariance
6. RMT-denoised covariance
7. Black-Litterman with basic AE-derived views
8. Black-Litterman with calibrated AE confidence + RMT covariance

**The AE -> BL bridge (the genuinely novel part):** for each latent-space cluster, compare its recent factor-mimicking return to its long-run average; form a mean-reversion view; then scale the confidence by cluster cohesion, signal-to-noise, sign stability, and reconstruction-error quality. This converts unsupervised structure into the P/Q/confidence triplet Black-Litterman needs without pretending the autoencoder directly discovers the final weights.

In [ ]:
def ae_views_for_bl(returns_d, clusters, lookback_recent=126, max_views=3):
    """Basic mechanical view generation: cluster-level mean reversion.
    Returns P (k x n), Q (annualized view returns), confidences."""
    P_rows, Q_vals, confs = [], [], []
    for c in sorted(clusters.unique()):
        members = clusters[clusters == c].index.intersection(returns_d.columns)
        if len(members) < 2:
            continue
        clu_r = returns_d[members].mean(axis=1).dropna()
        if len(clu_r) < 60:
            continue
        lb = min(lookback_recent, max(20, len(clu_r) // 2))
        recent = clu_r.tail(lb)
        recent_ann = recent.mean() * TRADING_DAYS
        longrun_ann = clu_r.mean() * TRADING_DAYS
        gap = longrun_ann - recent_ann
        row = np.zeros(len(returns_d.columns))
        idx = [returns_d.columns.get_loc(m) for m in members]
        row[idx] = 1 / len(members)
        P_rows.append(row)
        Q_vals.append(longrun_ann + 0.5 * gap)
        corr = returns_d[members].corr().values
        tri = np.triu_indices(len(members), 1)
        coh = np.nanmean(corr[tri]) if len(tri[0]) else 0.1
        confs.append(float(np.clip(max(coh, 0.1), 0.05, 0.95)))

    if len(P_rows) == 0:
        return np.empty((0, len(returns_d.columns))), np.array([]), np.array([])

    order = np.argsort(-np.abs(np.array(Q_vals)))[:max_views]
    return np.array(P_rows)[order], np.array(Q_vals)[order], np.array(confs)[order]


def black_litterman_given_sigma(Sigma, market_caps, P=None, Q=None, view_conf=None,
                                tau=0.05, delta=2.5):
    """Black-Litterman with an externally supplied annualized covariance matrix."""
    Sigma = clean_cov(Sigma)
    market_caps = pd.Series(market_caps, index=TICKERS if len(market_caps) == len(TICKERS) else None, dtype=float)
    market_caps = market_caps.reindex(TICKERS).fillna(market_caps.median()).values
    market_caps = np.nan_to_num(market_caps, nan=0.0, posinf=0.0, neginf=0.0)
    if market_caps.sum() <= 0:
        raise ValueError('market_caps must contain at least one positive value.')
    w_mkt = market_caps / market_caps.sum()
    pi = delta * Sigma @ w_mkt

    if P is None or len(P) == 0:
        return pi, Sigma

    P = np.atleast_2d(P)
    Q = np.atleast_1d(Q)
    view_conf = np.ones(len(Q)) if view_conf is None else np.asarray(view_conf, dtype=float)
    view_conf = np.clip(view_conf, 0.05, 0.99)

    Omega = np.diag(np.maximum(np.diag(P @ (tau * Sigma) @ P.T) / view_conf, 1e-10))
    inv_tau_sigma = np.linalg.pinv(tau * Sigma)
    inv_omega = np.linalg.pinv(Omega)
    M = np.linalg.pinv(inv_tau_sigma + P.T @ inv_omega @ P)
    mu_bl = M @ (inv_tau_sigma @ pi + P.T @ inv_omega @ Q)
    Sigma_bl = clean_cov(Sigma + M)
    return mu_bl, Sigma_bl


def _rolling_gap_flip_rate(cluster_returns, lookback=126, step=21):
    r = pd.Series(cluster_returns).dropna()
    if len(r) < 2 * lookback:
        return np.nan
    signs = []
    for end in range(2 * lookback, len(r) + 1, step):
        hist = r.iloc[:end]
        recent = r.iloc[end - lookback:end]
        gap = hist.mean() * TRADING_DAYS - recent.mean() * TRADING_DAYS
        signs.append(int(np.sign(gap)))
    signs = [s for s in signs if s != 0]
    if len(signs) < 2:
        return 0.0
    return float(np.mean([signs[i] != signs[i - 1] for i in range(1, len(signs))]))


def calibrated_ae_views_for_bl(returns_d, clusters, rec_error=None,
                               lookback_recent=126, max_views=5):
    """
    Calibrated AE -> BL view construction.

    Confidence combines:
    - cluster cohesion: do members really trade together?
    - signal-to-noise: is the mean-reversion gap large vs volatility?
    - sign stability: does the view direction keep flipping?
    - reconstruction quality: is the AE currently confused by the regime?
    """
    rows = []
    for c in sorted(clusters.unique()):
        members = clusters[clusters == c].index.intersection(returns_d.columns)
        if len(members) < 2:
            continue
        clu_r = returns_d[members].mean(axis=1).dropna()
        if len(clu_r) < 60:
            continue

        lb = int(min(lookback_recent, max(20, len(clu_r) // 2)))
        recent = clu_r.tail(lb)
        recent_ann = recent.mean() * TRADING_DAYS
        longrun_ann = clu_r.mean() * TRADING_DAYS
        gap_ann = longrun_ann - recent_ann
        q_ann = longrun_ann + 0.5 * gap_ann
        vol_ann = clu_r.std() * np.sqrt(TRADING_DAYS)
        gap_snr = abs(gap_ann) / (vol_ann + 1e-12)

        corr = returns_d[members].corr().values
        tri = np.triu_indices(len(members), 1)
        cohesion = np.nanmean(corr[tri]) if len(tri[0]) else np.nan
        cohesion = float(np.nan_to_num(cohesion, nan=0.0))
        cohesion_score = float(np.clip(cohesion / 0.85, 0.0, 1.0))
        snr_score = float(np.clip(gap_snr / 1.0, 0.0, 1.0))
        flip_rate = _rolling_gap_flip_rate(clu_r, lookback=lb, step=max(10, lb // 6))
        stability_score = 1.0 - (0.5 if np.isnan(flip_rate) else flip_rate)

        if rec_error is not None and len(pd.Series(rec_error).dropna()) > 0:
            er = pd.Series(rec_error).reindex(clu_r.index).dropna()
            all_er = pd.Series(rec_error).dropna()
            if len(er) > 10 and len(all_er) > 30:
                recent_er = er.tail(lb).mean()
                er_pct = float((all_er <= recent_er).mean())
                recon_score = 1.0 - er_pct
            else:
                recon_score = 0.5
        else:
            recon_score = 0.5

        confidence = 0.10 + 0.30 * cohesion_score + 0.30 * snr_score + 0.20 * stability_score + 0.10 * recon_score
        confidence = float(np.clip(confidence, 0.05, 0.95))
        fragility = float(np.clip(1.0 - confidence, 0.0, 1.0))

        row = np.zeros(len(returns_d.columns))
        idx = [returns_d.columns.get_loc(m) for m in members]
        row[idx] = 1.0 / len(members)

        rows.append({
            'Cluster': c,
            'Members': ', '.join(members.tolist()),
            'N': len(members),
            'LongRunAnn': longrun_ann,
            'RecentAnn': recent_ann,
            'GapAnn': gap_ann,
            'GapSign': int(np.sign(gap_ann)),
            'QAnn': q_ann,
            'VolAnn': vol_ann,
            'GapSNR': gap_snr,
            'Cohesion': cohesion,
            'FlipRate': flip_rate,
            'ReconScore': recon_score,
            'Confidence': confidence,
            'Fragility': fragility,
            'Score': abs(gap_ann) * confidence,
            'P_row': row,
        })

    if not rows:
        empty = pd.DataFrame()
        return np.empty((0, len(returns_d.columns))), np.array([]), np.array([]), empty

    view_table = pd.DataFrame(rows).set_index('Cluster').sort_values('Score', ascending=False)
    selected = view_table.head(max_views)
    P = np.vstack(selected['P_row'].values)
    Q = selected['QAnn'].values
    conf = selected['Confidence'].values
    return P, Q, conf, view_table.drop(columns=['P_row'])


P_ae, Q_ae, conf_ae = ae_views_for_bl(rets_d_full, clusters)
mu_blae, Sig_blae = black_litterman(rets_m_full, mcaps, P=P_ae, Q=Q_ae, view_conf=conf_ae)

P_cal, Q_cal, conf_cal, calibrated_views = calibrated_ae_views_for_bl(
    rets_d_full, clusters, rec_error=rec_err if 'rec_err' in globals() else None,
    lookback_recent=126, max_views=5
)
mu_bl_cal_rmt, Sig_bl_cal_rmt = black_litterman_given_sigma(
    Sig_rmt, mcaps.reindex(TICKERS), P=P_cal, Q=Q_cal, view_conf=conf_cal
)

print('Calibrated AE -> BL view table. Confidence is mechanical, not hand-picked.')
cols_to_show = ['Members', 'GapAnn', 'GapSNR', 'Cohesion', 'FlipRate', 'ReconScore', 'Confidence', 'Fragility', 'QAnn']
display(calibrated_views[cols_to_show].round(3))

risk_models = {
    'Sample cov':                 (mu_s, mu_sigma(rets_m_full)[1]),
    'Ledoit-Wolf':                (mu_s, Sig_lw),
    'PCA factor':                 (mu_s, pca_factor_cov(rets_m_full, periods=12)),
    'AE factor':                  (mu_s, Sig_ae),
    'DAE denoised':               (mu_s, Sig_dae),
    'RMT denoised':               (mu_s, Sig_rmt),
    'BL + AE views':              (mu_blae, Sig_blae),
    'BL calibrated AE + RMT':     (mu_bl_cal_rmt, Sig_bl_cal_rmt),
}

PORT_W = pd.DataFrame(index=TICKERS)
for name, (m, S) in risk_models.items():
    if name.startswith('BL'):
        PORT_W[name] = max_sharpe(m, S)
    else:
        PORT_W[name] = min_variance(m, S)

fig, ax = plt.subplots(figsize=(14, 4.5))
PORT_W.plot.bar(ax=ax, width=0.85)
ax.legend(fontsize=8, ncol=4)
ax.set_title('Weights by risk model, now including RMT and calibrated AE -> BL')
plt.tight_layout()
plt.show()

print('Concentration, network concentration, and top sector exposure:')
for c in PORT_W.columns:
    se = sector_exposure(PORT_W[c].values)
    net_score = network_concentration_score(PORT_W[c].values) if 'network_concentration_score' in globals() else np.nan
    print(f'  {c:24s} HHI={herfindahl(PORT_W[c]):.3f} | NetConc={net_score:.2f} | top: {se.index[0]} {se.iloc[0]:.0%}, {se.index[1]} {se.iloc[1]:.0%}')


## 8. Walk-forward backtest

A useful rolling OOS diagnostic, but **not** the final clean hold-out test. Protocol:

- **Rolling 36-month training window** → estimate the risk model → optimize → **hold for one quarter** → roll forward. Weights at time *t* use only data through *t*.
- **Autoencoders retrained inside each window** (smaller/faster config) — no peeking.
- **Costs:** 10 bps × one-way turnover deducted at each rebalance.
- **Drift:** weights drift with prices between rebalances (no continuous rebalancing fantasy).
- Universe starts when all 18 names trade (mid-2019) so the panel is complete — which further shortens an already-short test and means the backtest covers essentially **one bull regime plus 2020 and 2022**. Statistical power is low; treat rankings as suggestive. The next section adds the stricter train/validate/test split so the final test period is not touched during model development.

What this still **cannot** fix: the hindsight-selected universe. Every strategy below will likely beat SPY. The *relative* ranking across risk models is the meaningful output; the absolute outperformance is not.

In [ ]:
def train_window_ae(X_win, k, denoise=False, epochs=150):
    """Lightweight AE retrain inside each walk-forward window."""
    sp = int(len(X_win)*0.85)
    m, _ = train_model(AE(X_win.shape[1], k, hidden=24), X_win[:sp], X_win[sp:],
                       epochs=epochs, noise_std=0.5 if denoise else 0.0, patience=20)
    return m

def window_clusters(win_d, Zw, n_clusters=5):
    """Cluster assets using ONLY this window's AE latent factors -- the
    Section-5 clustering was fit once on the full sample, which would leak
    future structure into a backtest if reused as-is. This recomputes it
    fresh inside every rolling window, same discipline as AE factor / DAE."""
    n_c = min(n_clusters, win_d.shape[1] - 1)
    emb = pd.DataFrame(
        [[np.corrcoef(win_d[t].values, Zw[:, j])[0, 1] for j in range(Zw.shape[1])]
         for t in win_d.columns],
        index=win_d.columns)
    km = KMeans(n_clusters=n_c, n_init=10, random_state=SEED).fit(emb.values)
    return pd.Series(km.labels_, index=win_d.columns)

def estimate_sigma(model_name, win_d, win_m, mcaps_w=None):
    """Dispatch: risk model name -> (mu, Sigma) using ONLY the window data."""
    mu_w = win_m.mean().values * 12
    if model_name == "Sample cov":   return mu_w, win_m.cov().values * 12
    if model_name == "Ledoit-Wolf":  return mu_w, lw_sigma(win_m)
    if model_name == "RMT denoised": return mu_w, rmt_denoised_cov(win_m, periods=12)
    if model_name == "PCA factor":   return mu_w, pca_factor_cov(win_m, k=min(K_FACTORS, 5), periods=12)
    sc = StandardScaler(); Xw = sc.fit_transform(win_d.values)
    if model_name == "AE factor":
        m = train_window_ae(Xw, K_FACTORS)
        with torch.no_grad():
            Zw = m.encoder(torch.tensor(Xw, dtype=torch.float32, device=DEVICE)).cpu().numpy()
        return mu_w, ae_factor_cov(win_d, Zw)
    if model_name == "DAE denoised":
        m = train_window_ae(Xw, K_FACTORS, denoise=True)
        return mu_w, dae_denoised_cov(win_d, m, sc)
    if model_name == "BL + AE views":
        m = train_window_ae(Xw, K_FACTORS)
        with torch.no_grad():
            Zw = m.encoder(torch.tensor(Xw, dtype=torch.float32, device=DEVICE)).cpu().numpy()
        clusters_w = window_clusters(win_d, Zw)
        P_w, Q_w, conf_w = ae_views_for_bl(win_d, clusters_w,
                                            lookback_recent=min(126, len(win_d) // 2))
        if len(P_w) == 0 or mcaps_w is None:
            # no usable clusters this window (or no mcaps available) -> fall
            # back to the prior with no tilt, rather than silently crashing
            return mu_w, lw_sigma(win_m)
        return black_litterman(win_m, mcaps_w, P=P_w, Q=Q_w, view_conf=conf_w)
    if model_name == "BL calibrated AE + RMT":
        m = train_window_ae(Xw, K_FACTORS)
        with torch.no_grad():
            Zw = m.encoder(torch.tensor(Xw, dtype=torch.float32, device=DEVICE)).cpu().numpy()
        clusters_w = window_clusters(win_d, Zw)
        P_w, Q_w, conf_w, _ = calibrated_ae_views_for_bl(
            win_d, clusters_w, rec_error=None,
            lookback_recent=min(126, len(win_d) // 2), max_views=5
        )
        Sig_rmt_w = rmt_denoised_cov(win_m, periods=12)
        if len(P_w) == 0 or mcaps_w is None:
            return mu_w, Sig_rmt_w
        return black_litterman_given_sigma(Sig_rmt_w, mcaps_w, P=P_w, Q=Q_w, view_conf=conf_w)
    raise ValueError(model_name)

def walk_forward(model_name, daily, monthly, train_months=36, rebal_months=3,
                 tc=TC_BPS/1e4, equal_weight=False, mcaps_w=None):
    months = monthly.index
    dates, weights = [], []
    i = train_months
    while i < len(months):
        t_end = months[i-1]
        win_m = monthly.iloc[i-train_months:i]
        win_d = daily.loc[:t_end].tail(train_months*21)
        if equal_weight:
            w = np.ones(len(TICKERS))/len(TICKERS)
        else:
            mu_w, Sig_w = estimate_sigma(model_name, win_d, win_m, mcaps_w=mcaps_w)
            # BL posterior is a return forecast, not just a risk model -- use it
            # the same way Section 7 does (max-Sharpe), everything else stays
            # the risk-only min-variance core.
            w = max_sharpe(mu_w, Sig_w) if model_name in ("BL + AE views", "BL calibrated AE + RMT") else min_variance(mu_w, Sig_w)
        dates.append(months[i]); weights.append(w)
        i += rebal_months
    # Simulate daily with drift + costs
    port_val, w_cur, ret_list, ridx = 1.0, None, [], []
    rebal_map = dict(zip(dates, weights))
    for d in daily.index:
        # apply rebalance scheduled at month-start AFTER the signal month-end
        due = [k for k in rebal_map if k <= d]
        if due:
            target = rebal_map.pop(max(due))
            if w_cur is None:
                w_cur, cost = np.array(target), tc * 1.0
            else:
                cost = tc * np.abs(np.array(target) - w_cur).sum()
                w_cur = np.array(target)
            port_val *= (1 - cost)
        if w_cur is None: continue
        r = np.nansum(w_cur * daily.loc[d].values)
        port_val *= (1 + r)
        w_cur = w_cur * (1 + daily.loc[d].values)      # drift
        w_cur = w_cur / w_cur.sum()
        ret_list.append(r); ridx.append(d)
    rets = pd.Series(ret_list, index=ridx, name=model_name)
    return rets

BT_MODELS = ["Sample cov", "Ledoit-Wolf", "RMT denoised", "PCA factor", "AE factor", "DAE denoised", "BL + AE views", "BL calibrated AE + RMT"]
bt = {}
print("Running walk-forward backtests (AE models retrain per window — a few minutes on CPU)...")
for mname in BT_MODELS:
    bt[mname] = walk_forward(mname, rets_d_full, rets_m_full, mcaps_w=mcaps)
    print(f"  done: {mname}")
bt["Equal weight"] = walk_forward("EW", rets_d_full, rets_m_full, equal_weight=True)
bt_start = bt["Equal weight"].index[0]
bt["SPY"] = bench_d["SPY"].loc[bt_start:]
bt["QQQ"] = bench_d["QQQ"].loc[bt_start:]


In [ ]:
# Portfolio value from €1,000 + full comparison table
fig, ax = plt.subplots(figsize=(12, 6))
for n, r in bt.items():
    (INITIAL_CAPITAL_EUR * (1 + r).cumprod()).plot(ax=ax, label=n,
        lw=2.2 if n in ("SPY", "QQQ") else 1.2,
        ls="--" if n in ("SPY", "QQQ") else "-")
ax.set_title("Walk-forward, net of 10bps costs: €1,000 →  (out-of-sample weights, hindsight universe)")
ax.set_ylabel("EUR"); ax.legend(fontsize=8); plt.show()

# Turnover per strategy (re-derive from rebalance simulation is complex; approximate via weight series)
tbl = perf_table(bt)
tbl["Final € (from 1,000)"] = [round(INITIAL_CAPITAL_EUR * float((1 + bt[n]).prod()), 0) for n in tbl.index]
tbl.sort_values("Sharpe", ascending=False)


In [ ]:
# Stress-period zoom + rolling diagnostics
stress_windows = {"COVID crash": ("2020-02-15", "2020-04-30"),
                  "2022 hike cycle": ("2022-01-01", "2022-12-31")}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax_, (nm, (a, b)) in zip(axes, stress_windows.items()):
    for n in ["Ledoit-Wolf", "AE factor", "DAE denoised", "BL + AE views", "Equal weight", "SPY"]:
        seg = bt[n].loc[a:b]
        if len(seg) == 0: continue
        (1 + seg).cumprod().plot(ax=ax_, label=n, lw=1.2)
    ax_.set_title(f"Stress test: {nm}"); ax_.legend(fontsize=7)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for n in ["Ledoit-Wolf", "AE factor", "BL + AE views", "Equal weight", "SPY"]:
    (bt[n].rolling(63).std()*np.sqrt(252)).plot(ax=axes[0], label=n, lw=1)
    ((bt[n].rolling(252).mean()*252 - RF_ANNUAL) /
     (bt[n].rolling(252).std()*np.sqrt(252))).plot(ax=axes[1], label=n, lw=1)
axes[0].set_title("Rolling 3-month volatility (annualized)"); axes[0].legend(fontsize=8)
axes[1].set_title("Rolling 1-year Sharpe"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(12, 3.5))
for n in ["Ledoit-Wolf", "AE factor", "BL + AE views", "Equal weight", "SPY"]:
    drawdown_series(bt[n]).plot(ax=ax, label=n, lw=1)
ax.set_title("Drawdowns, walk-forward"); ax.legend(fontsize=8); plt.show()


## 8B. True chronological hold-out: train / validate / untouched final test

This is the stricter OOS discipline that the rolling walk-forward section does **not** fully provide.

Protocol:

- **Train:** pre-2020 data only.
- **Validate:** 2020-2022. Use this period for model selection and sanity checks.
- **Test:** 2023-present. This period is treated as the final hold-out and should not influence model choice, hyperparameters, view rules, or interpretation until the end.

Important caveat: the full 18-stock panel starts only when every name has history. Because CRWD IPO'd in 2019, a strict all-18 pre-2020 training set is too short to be meaningful. This section therefore reports that failure explicitly and then runs a clean hold-out on the largest sub-universe with sufficient pre-2020 history. That is more honest than pretending the all-18 notebook already had a clean final test.

In [ ]:
# --- True chronological hold-out: train / validate / final untouched test ---

HOLDOUT_TRAIN_END = pd.Timestamp('2019-12-31')
HOLDOUT_VALID_START = pd.Timestamp('2020-01-01')
HOLDOUT_VALID_END = pd.Timestamp('2022-12-31')
HOLDOUT_TEST_START = pd.Timestamp('2023-01-01')

HOLDOUT_MIN_TRAIN_MONTHS = 36
HOLDOUT_MIN_VALID_MONTHS = 24
HOLDOUT_MIN_TEST_MONTHS = 12
HOLDOUT_CANDIDATES = ['Equal weight', 'Ledoit-Wolf', 'RMT denoised', 'BL calibrated AE + RMT']


def _safe_date(x):
    return pd.Timestamp(x).date() if pd.notna(x) else None


def holdout_universe_diagnostics(rets_m_all, tickers):
    rows = []
    for t in tickers:
        s = rets_m_all[t]
        train_n = s.loc[:HOLDOUT_TRAIN_END].dropna().shape[0]
        valid_n = s.loc[HOLDOUT_VALID_START:HOLDOUT_VALID_END].dropna().shape[0]
        test_n = s.loc[HOLDOUT_TEST_START:].dropna().shape[0]
        rows.append({
            'Ticker': t,
            'FirstMonthlyReturn': _safe_date(s.dropna().index.min()) if len(s.dropna()) else None,
            'TrainMonths_pre2020': train_n,
            'ValidationMonths_2020_2022': valid_n,
            'TestMonths_2023_present': test_n,
            'EligibleCleanHoldout': (
                train_n >= HOLDOUT_MIN_TRAIN_MONTHS and
                valid_n >= HOLDOUT_MIN_VALID_MONTHS and
                test_n >= HOLDOUT_MIN_TEST_MONTHS
            ),
        })
    return pd.DataFrame(rows).set_index('Ticker')


holdout_universe_check = holdout_universe_diagnostics(rets_m, TICKERS)
eligible_holdout_tickers = holdout_universe_check.index[holdout_universe_check['EligibleCleanHoldout']].tolist()
excluded_holdout_tickers = [t for t in TICKERS if t not in eligible_holdout_tickers]

strict_all18_train_months = rets_m_full.loc[:HOLDOUT_TRAIN_END].shape[0]
print(f'Strict all-18 complete-panel training months before 2020: {strict_all18_train_months}')
if strict_all18_train_months < HOLDOUT_MIN_TRAIN_MONTHS:
    print('Result: FAIL for a strict all-18 pre-2020 hold-out. The complete 18-name panel starts too late.')

print(f'Clean hold-out universe: {len(eligible_holdout_tickers)} names')
print(f'Excluded from clean hold-out due insufficient pre-2020/validation/test history: {excluded_holdout_tickers}')
display(holdout_universe_check.sort_values(['EligibleCleanHoldout', 'TrainMonths_pre2020'], ascending=[True, True]))

if len(eligible_holdout_tickers) < 6:
    raise ValueError('Too few eligible tickers for a meaningful clean hold-out test.')

holdout_d = rets_d[eligible_holdout_tickers].dropna(how='any')
holdout_m = rets_m[eligible_holdout_tickers].dropna(how='any')

holdout_train_d = holdout_d.loc[:HOLDOUT_TRAIN_END]
holdout_valid_d = holdout_d.loc[HOLDOUT_VALID_START:HOLDOUT_VALID_END]
holdout_test_d = holdout_d.loc[HOLDOUT_TEST_START:]

holdout_train_m = holdout_m.loc[:HOLDOUT_TRAIN_END]
holdout_valid_m = holdout_m.loc[HOLDOUT_VALID_START:HOLDOUT_VALID_END]
holdout_test_m = holdout_m.loc[HOLDOUT_TEST_START:]

holdout_split_summary = pd.DataFrame({
    'DailyObs': [len(holdout_train_d), len(holdout_valid_d), len(holdout_test_d)],
    'MonthlyObs': [len(holdout_train_m), len(holdout_valid_m), len(holdout_test_m)],
    'Start': [
        _safe_date(holdout_train_d.index.min()),
        _safe_date(holdout_valid_d.index.min()),
        _safe_date(holdout_test_d.index.min()),
    ],
    'End': [
        _safe_date(holdout_train_d.index.max()),
        _safe_date(holdout_valid_d.index.max()),
        _safe_date(holdout_test_d.index.max()),
    ],
}, index=['Train_pre2020', 'Validate_2020_2022', 'FinalTest_2023_present'])
print('Chronological split actually used for the clean hold-out:')
display(holdout_split_summary)


# Local BL implementation that works on a sub-universe instead of assuming the global TICKERS list.
def black_litterman_given_sigma_subset(Sigma, market_caps, columns, P=None, Q=None, view_conf=None,
                                       tau=0.05, delta=2.5):
    Sigma = clean_cov(Sigma)
    columns = list(columns)
    caps = pd.Series(market_caps, dtype=float).reindex(columns)
    if caps.isna().all() or caps.fillna(0).sum() <= 0:
        caps = pd.Series(1.0, index=columns)
    else:
        caps = caps.fillna(caps.median())
    w_mkt = caps.values / caps.values.sum()
    pi = delta * Sigma @ w_mkt

    if P is None or len(P) == 0:
        return pi, Sigma

    P = np.atleast_2d(P)
    Q = np.atleast_1d(Q)
    view_conf = np.ones(len(Q)) if view_conf is None else np.asarray(view_conf, dtype=float)
    view_conf = np.clip(view_conf, 0.05, 0.99)

    Omega = np.diag(np.maximum(np.diag(P @ (tau * Sigma) @ P.T) / view_conf, 1e-10))
    inv_tau_sigma = np.linalg.pinv(tau * Sigma)
    inv_omega = np.linalg.pinv(Omega)
    M = np.linalg.pinv(inv_tau_sigma + P.T @ inv_omega @ P)
    mu_bl = M @ (inv_tau_sigma @ pi + P.T @ inv_omega @ Q)
    Sigma_bl = clean_cov(Sigma + M)
    return mu_bl, Sigma_bl


def _holdout_ae_clusters_and_recon(train_d, k=None, epochs=150):
    k = min(K_FACTORS if k is None else k, max(1, train_d.shape[1] - 1))
    sc = StandardScaler()
    X_train = sc.fit_transform(train_d.values)
    split = max(30, int(len(X_train) * 0.85))
    split = min(split, len(X_train) - 20)
    model, _ = train_model(
        AE(train_d.shape[1], k, hidden=24),
        X_train[:split], X_train[split:],
        epochs=epochs, patience=20
    )
    model.eval()
    Xt = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        rec, z = model(Xt)
    Z = z.cpu().numpy()
    rec_err_train = pd.Series(((rec.cpu().numpy() - X_train) ** 2).mean(axis=1), index=train_d.index)
    clusters_train = window_clusters(train_d, Z, n_clusters=min(5, train_d.shape[1] - 1))
    return clusters_train, rec_err_train


def fit_holdout_weights(model_name, train_d, train_m, tickers, market_caps):
    tickers = list(tickers)
    max_w = max(0.20, 1.0 / len(tickers) + 1e-6)
    mu_train = train_m.mean().values * 12

    if model_name == 'Equal weight':
        return pd.Series(np.ones(len(tickers)) / len(tickers), index=tickers), None

    if model_name == 'Ledoit-Wolf':
        Sigma = lw_sigma(train_m, periods=12)
        w = min_variance(mu_train, Sigma, max_w=max_w)
        return pd.Series(w, index=tickers), None

    if model_name == 'RMT denoised':
        Sigma = rmt_denoised_cov(train_m, periods=12)
        w = min_variance(mu_train, Sigma, max_w=max_w)
        return pd.Series(w, index=tickers), None

    if model_name == 'BL calibrated AE + RMT':
        clusters_train, rec_err_train = _holdout_ae_clusters_and_recon(train_d, epochs=150)
        P, Q, conf, view_table = calibrated_ae_views_for_bl(
            train_d, clusters_train, rec_error=rec_err_train,
            lookback_recent=min(126, max(40, len(train_d) // 3)), max_views=5
        )
        Sigma_rmt = rmt_denoised_cov(train_m, periods=12)
        if len(P) == 0:
            mu_bl, Sigma_bl = mu_train, Sigma_rmt
        else:
            mu_bl, Sigma_bl = black_litterman_given_sigma_subset(
                Sigma_rmt, market_caps, tickers, P=P, Q=Q, view_conf=conf
            )
        try:
            w = max_sharpe(mu_bl, Sigma_bl, max_w=max_w)
        except Exception:
            w = min_variance(mu_bl, Sigma_bl, max_w=max_w)
        return pd.Series(w, index=tickers), view_table

    raise ValueError(f'Unknown hold-out candidate: {model_name}')


def static_weight_returns(daily_returns, weights, tc=TC_BPS / 1e4):
    """One initial rebalance, then weights drift with asset returns. No test-period refits."""
    daily_returns = pd.DataFrame(daily_returns).dropna(how='any')
    w_cur = pd.Series(weights, index=daily_returns.columns).fillna(0.0).values.astype(float)
    w_cur = np.clip(w_cur, 0, None)
    w_cur = w_cur / w_cur.sum()

    out = []
    first = True
    for dt, row in daily_returns.iterrows():
        asset_r = row.values.astype(float)
        gross_r = float(np.nansum(w_cur * asset_r))
        net_r = (1 - tc) * (1 + gross_r) - 1 if first else gross_r
        first = False
        out.append((dt, net_r))
        w_cur = w_cur * (1 + asset_r)
        if w_cur.sum() > 0:
            w_cur = w_cur / w_cur.sum()
    return pd.Series(dict(out)).sort_index()


# 1) Train on pre-2020 only, evaluate on validation, and choose one primary model.
holdout_mcaps = mcaps.reindex(eligible_holdout_tickers).fillna(mcaps.median()) if 'mcaps' in globals() else pd.Series(1.0, index=eligible_holdout_tickers)
holdout_validation_returns = {}
holdout_validation_weights = {}
holdout_validation_views = {}

for candidate in HOLDOUT_CANDIDATES:
    w, vt = fit_holdout_weights(candidate, holdout_train_d, holdout_train_m, eligible_holdout_tickers, holdout_mcaps)
    holdout_validation_weights[candidate] = w
    holdout_validation_views[candidate] = vt
    holdout_validation_returns[candidate] = static_weight_returns(holdout_valid_d, w)

true_holdout_validation = perf_table(holdout_validation_returns)
true_holdout_validation['Final € from 1,000'] = [
    INITIAL_CAPITAL_EUR * float((1 + holdout_validation_returns[n]).prod()) for n in true_holdout_validation.index
]
true_holdout_validation = true_holdout_validation.sort_values('Sharpe', ascending=False)
print('Validation period only: 2020-2022. Model choice is made here, before seeing 2023-present.')
display(true_holdout_validation.round(3))

true_holdout_primary_model = true_holdout_validation.index[0]
print(f'Primary model selected from validation only: {true_holdout_primary_model}')


# 2) Freeze the selection rule, refit each pre-registered candidate on train+validation, then evaluate 2023-present once.
holdout_train_valid_d = holdout_d.loc[:HOLDOUT_VALID_END]
holdout_train_valid_m = holdout_m.loc[:HOLDOUT_VALID_END]

true_holdout_weights = {}
true_holdout_views = {}
true_holdout_test_returns = {}

for candidate in HOLDOUT_CANDIDATES:
    w, vt = fit_holdout_weights(candidate, holdout_train_valid_d, holdout_train_valid_m, eligible_holdout_tickers, holdout_mcaps)
    true_holdout_weights[candidate] = w
    true_holdout_views[candidate] = vt
    true_holdout_test_returns[candidate] = static_weight_returns(holdout_test_d, w)

true_holdout_test = perf_table(true_holdout_test_returns)
true_holdout_test['Final € from 1,000'] = [
    INITIAL_CAPITAL_EUR * float((1 + true_holdout_test_returns[n]).prod()) for n in true_holdout_test.index
]
true_holdout_test['PrimarySelectedOnValidation'] = true_holdout_test.index == true_holdout_primary_model
true_holdout_test = true_holdout_test.sort_values(['PrimarySelectedOnValidation', 'Sharpe'], ascending=[False, False])

print('FINAL UNTOUCHED TEST: 2023-present. Do not use this table to redesign the model.')
display(true_holdout_test.round(3))

fig, ax = plt.subplots(figsize=(12, 5))
for name, r in true_holdout_test_returns.items():
    lw = 2.4 if name == true_holdout_primary_model else 1.1
    (INITIAL_CAPITAL_EUR * (1 + r).cumprod()).plot(ax=ax, label=name, lw=lw)
ax.set_title('True chronological hold-out: final test only, 2023-present')
ax.set_ylabel('EUR from initial 1,000')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

primary_test = true_holdout_test.loc[true_holdout_primary_model]
print(f"Primary validation-selected model on final test: {true_holdout_primary_model}")
print(f"Test Sharpe: {primary_test['Sharpe']:.3f} | Test MaxDD: {primary_test['MaxDD']:.1%} | Final EUR: {primary_test['Final € from 1,000']:.0f}")
print('Interpretation rule: this table is final evidence, not another tuning surface.')

### Regime-aware AE -> Black-Litterman view testing

A view engine that only works in calm markets is not robust. This section checks whether the AE-derived view signs, confidence scores, and top clusters change across regimes defined by volatility, drawdown, momentum, and AE reconstruction-error stress.

In [ ]:
# 5) Regime-aware view testing

def build_regime_labels(returns_d, stress_flag=None):
    port = returns_d.mean(axis=1).dropna()
    vol21 = port.rolling(21).std() * np.sqrt(TRADING_DAYS)
    mom63 = port.rolling(63).mean() * TRADING_DAYS
    dd = drawdown_series(port)

    labels = pd.Series('normal', index=port.index)
    vol_hi = vol21.quantile(0.75)
    vol_lo = vol21.quantile(0.35)

    labels[(mom63 > 0) & (vol21 < vol_lo) & (dd > -0.08)] = 'calm_uptrend'
    labels[(dd < -0.15) | (port.rolling(21).sum() < -0.10)] = 'drawdown'
    labels[vol21 > vol_hi] = 'high_vol'
    if stress_flag is not None:
        sf = pd.Series(stress_flag).reindex(labels.index).fillna(0).astype(bool)
        labels[sf] = 'ae_stress'
    return labels


regime_labels = build_regime_labels(rets_d_full, stress if 'stress' in globals() else None)
regime_counts = regime_labels.value_counts().rename('Days')
print('Regime counts:')
display(regime_counts.to_frame())

baseline_signs = calibrated_views['GapSign'].rename('FullSampleGapSign') if len(calibrated_views) else pd.Series(dtype=float)
regime_rows = []
regime_view_tables = {}
for regime_name, dates in regime_labels.groupby(regime_labels).groups.items():
    sub = rets_d_full.loc[list(dates)].dropna(how='any')
    if len(sub) < 60:
        continue
    lb = min(63, max(20, len(sub) // 2))
    _, _, _, tbl_r = calibrated_ae_views_for_bl(
        sub, clusters, rec_error=rec_err if 'rec_err' in globals() else None,
        lookback_recent=lb, max_views=5
    )
    if tbl_r.empty:
        continue
    regime_view_tables[regime_name] = tbl_r
    joined = tbl_r[['GapSign']].join(baseline_signs, how='inner')
    mask = joined['FullSampleGapSign'] != 0
    sign_flip_vs_full = float((joined.loc[mask, 'GapSign'] != joined.loc[mask, 'FullSampleGapSign']).mean()) if mask.any() else np.nan
    regime_rows.append({
        'Regime': regime_name,
        'Days': len(sub),
        'TopCluster': tbl_r.index[0],
        'TopMembers': tbl_r.iloc[0]['Members'],
        'MedianConfidence': tbl_r['Confidence'].median(),
        'MedianFragility': tbl_r['Fragility'].median(),
        'MedianGapSNR': tbl_r['GapSNR'].median(),
        'SignFlipRateVsFullSample': sign_flip_vs_full,
    })

regime_view_summary = pd.DataFrame(regime_rows).set_index('Regime').sort_values('MedianFragility', ascending=False)
print('Regime-aware view stability summary:')
display(regime_view_summary.round(3))

fig, ax = plt.subplots(figsize=(10, 4))
regime_counts.plot.bar(ax=ax)
ax.set_title('Days per detected regime')
ax.set_ylabel('Trading days')
plt.tight_layout()
plt.show()

print('Interpretation: high sign flips or low confidence in stress regimes means the AE -> BL overlay should be capped or switched off there.')

## 9. Synthetic stress-testing: does AE → Black-Litterman survive alternate histories?

Iñaki's most useful suggestion is to treat synthetic data as a **fragility test** for the view-construction engine.

The question is not, "Can synthetic data prove this strategy works?" It cannot. The question is narrower and more useful:

> If we perturb the history through bootstraps, high-volatility bear regimes, correlation spikes, and cluster rotations, do the AE-derived BL views stay directionally coherent, or do they flip around?

This section adds three diagnostics:

1. **View stability** — how often the dominant cluster view changes, and how often view signs flip versus the real-history view.
2. **Signal-to-noise** — whether the mean-reversion gap is large relative to sampling noise, not just numerically large.
3. **Posterior-weight fragility** — whether small changes in views cause the BL portfolio to jump far away from the simpler Ledoit-Wolf core.

Interpretation rule: synthetic tests can **falsify** a fragile signal, but they do not validate investability. The OOS sample-size section below remains the gating issue.


In [ ]:
# --- Synthetic stress-test helpers for AE -> Black-Litterman views ---

def cluster_view_table(returns_d, clusters, lookback_recent=126):
    """
    Transparent version of ae_views_for_bl().
    It keeps every cluster-level view, not just the top max_views rows.

    Directional signal = GapAnn:
      positive  -> recent return < long-run return -> mean-reversion catch-up view
      negative  -> recent return > long-run return -> fade the recent winner
    """
    rows = []
    for c in sorted(clusters.unique()):
        members = clusters[clusters == c].index.intersection(returns_d.columns)
        if len(members) < 2:
            continue

        clu_r = returns_d[members].mean(axis=1).dropna()
        if len(clu_r) < 60:
            continue

        lb = int(min(lookback_recent, max(20, len(clu_r) // 2)))
        recent = clu_r.tail(lb)

        recent_ann = recent.mean() * TRADING_DAYS
        longrun_ann = clu_r.mean() * TRADING_DAYS
        gap_ann = longrun_ann - recent_ann
        q_ann = longrun_ann + 0.5 * gap_ann

        # Annualized standard error of the recent mean. This is deliberately rough:
        # it is here to stop us from over-reading tiny mean differences.
        se_recent_ann = recent.std() / np.sqrt(len(recent)) * TRADING_DAYS
        gap_t = gap_ann / se_recent_ann if se_recent_ann > 0 else np.nan

        corr = returns_d[members].corr().values
        tri = np.triu_indices(len(members), 1)
        coherence = np.nanmean(corr[tri]) if len(tri[0]) else np.nan

        rows.append({
            "Cluster": c,
            "Members": ", ".join(members.tolist()),
            "N": len(members),
            "LongRunAnn": longrun_ann,
            "RecentAnn": recent_ann,
            "GapAnn": gap_ann,
            "GapSign": int(np.sign(gap_ann)),
            "QAnn": q_ann,
            "AbsQ": abs(q_ann),
            "AbsGap": abs(gap_ann),
            "GapT": gap_t,
            "Coherence": max(float(coherence), 0.1) if np.isfinite(coherence) else 0.1,
        })

    if not rows:
        return pd.DataFrame()

    out = pd.DataFrame(rows).set_index("Cluster")
    return out.sort_values("AbsQ", ascending=False)


SYNTH_SCENARIOS = {
    # Preserves empirical blocks, dependence, and many tail events.
    "empirical_blocks": {
        "vol_mult": 1.00, "corr_blend": 0.00, "drift_ann": 0.00,
        "jump_prob": 0.00,
    },

    # A benign regime: lower realized volatility, lower cross-sectional comovement.
    "calm_low_vol": {
        "vol_mult": 0.65, "corr_blend": -0.10, "drift_ann": 0.04,
        "jump_prob": 0.00,
    },

    # A hostile regime: negative drift, higher vol, higher correlation, occasional market jumps.
    "bear_high_corr": {
        "vol_mult": 1.80, "corr_blend": 0.55, "drift_ann": -0.18,
        "jump_prob": 0.015, "jump_mean": -0.045, "jump_sd": 0.015,
    },

    # A cross-sectional regime: one latent cluster wins first, then loses, and another does the opposite.
    "cluster_rotation": {
        "vol_mult": 1.15, "corr_blend": 0.15, "drift_ann": 0.00,
        "jump_prob": 0.005, "jump_mean": -0.030, "jump_sd": 0.010,
        "cluster_rotation": True, "cluster_shock_ann": 0.18,
    },
}


def moving_block_bootstrap(returns_d, n_days=3 * TRADING_DAYS, block_len=21, seed=SEED):
    """
    Bootstrap contiguous daily blocks. This keeps short-run autocorrelation,
    volatility clustering, and same-day cross-asset dependence better than iid sampling.
    """
    rng = np.random.default_rng(seed)
    R = returns_d.dropna()
    if len(R) <= block_len:
        raise ValueError("Need more observations than block_len for block bootstrap.")

    starts = rng.integers(0, len(R) - block_len + 1,
                          size=int(np.ceil(n_days / block_len)))
    arr = np.vstack([R.iloc[s:s + block_len].values for s in starts])[:n_days]
    idx = pd.bdate_range("2030-01-03", periods=n_days)
    return pd.DataFrame(arr, index=idx, columns=R.columns)


def _apply_scenario_transform(path_d, returns_d, clusters, scenario, seed=SEED):
    """Apply scenario-level drift / vol / correlation / cluster-rotation transforms."""
    if scenario not in SYNTH_SCENARIOS:
        raise ValueError(f"Unknown scenario: {scenario}")

    rng = np.random.default_rng(seed)
    cfg = SYNTH_SCENARIOS[scenario]

    X = path_d.values.copy()
    mu = returns_d.mean().reindex(path_d.columns).values
    centered = X - mu

    # Increase/decrease common-factor dominance.
    market_component = centered.mean(axis=1, keepdims=True)
    corr_blend = cfg.get("corr_blend", 0.0)
    if corr_blend >= 0:
        transformed = (1 - corr_blend) * centered + corr_blend * market_component
    else:
        # Negative corr_blend lightly restores idiosyncratic behavior in calm regimes.
        transformed = centered - abs(corr_blend) * market_component

    X_new = mu + cfg.get("vol_mult", 1.0) * transformed
    X_new += cfg.get("drift_ann", 0.0) / TRADING_DAYS

    if cfg.get("cluster_rotation", False):
        cluster_order = clusters.value_counts().index.tolist()
        if len(cluster_order) >= 2:
            c_a, c_b = cluster_order[0], cluster_order[1]
            shock = cfg.get("cluster_shock_ann", 0.18) / TRADING_DAYS
            half = len(path_d) // 2

            cols_a = [path_d.columns.get_loc(t) for t in clusters[clusters == c_a].index
                      if t in path_d.columns]
            cols_b = [path_d.columns.get_loc(t) for t in clusters[clusters == c_b].index
                      if t in path_d.columns]

            # First half: A wins, B loses. Second half: reversal.
            X_new[:half, cols_a] += shock
            X_new[:half, cols_b] -= shock
            X_new[half:, cols_a] -= shock
            X_new[half:, cols_b] += shock

    if cfg.get("jump_prob", 0.0) > 0:
        jumps = rng.random(len(path_d)) < cfg["jump_prob"]
        if jumps.any():
            common_jump = rng.normal(cfg.get("jump_mean", -0.04),
                                     cfg.get("jump_sd", 0.01),
                                     size=(jumps.sum(), 1))
            X_new[jumps, :] += common_jump

    # Avoid impossible returns from extreme synthetic transforms.
    X_new = np.clip(X_new, -0.95, 1.00)
    return pd.DataFrame(X_new, index=path_d.index, columns=path_d.columns)


def make_regime_switching_path(returns_d, clusters, n_days=3 * TRADING_DAYS,
                               block_len=21, seed=SEED):
    """
    Markov-style regime mixture built from the four scenario transforms above.
    The goal is not realism; it is a controlled alternate history with regime changes.
    """
    rng = np.random.default_rng(seed)
    regimes = ["calm_low_vol", "empirical_blocks", "bear_high_corr", "cluster_rotation"]

    # Rows = current state, columns = next state.
    P = np.array([
        [0.70, 0.20, 0.05, 0.05],  # calm tends to persist
        [0.20, 0.55, 0.15, 0.10],  # normal can go anywhere
        [0.10, 0.20, 0.55, 0.15],  # stress clusters
        [0.10, 0.25, 0.15, 0.50],  # rotations persist but mean-revert
    ])

    chunks, state = [], 1
    for k in range(int(np.ceil(n_days / block_len))):
        state = rng.choice(len(regimes), p=P[state])
        raw = moving_block_bootstrap(returns_d, n_days=block_len, block_len=block_len,
                                     seed=int(rng.integers(0, 1_000_000)))
        tr = _apply_scenario_transform(raw, returns_d, clusters, regimes[state],
                                       seed=int(rng.integers(0, 1_000_000)))
        chunks.append(tr.values)

    arr = np.vstack(chunks)[:n_days]
    idx = pd.bdate_range("2040-01-03", periods=n_days)
    return pd.DataFrame(arr, index=idx, columns=returns_d.columns)


def make_synthetic_path(returns_d, clusters, scenario="empirical_blocks",
                        n_days=3 * TRADING_DAYS, block_len=21, seed=SEED):
    if scenario == "regime_switching":
        return make_regime_switching_path(returns_d, clusters, n_days=n_days,
                                          block_len=block_len, seed=seed)

    raw = moving_block_bootstrap(returns_d, n_days=n_days, block_len=block_len, seed=seed)
    return _apply_scenario_transform(raw, returns_d, clusters, scenario, seed=seed)


def normalized_entropy(x):
    """0 = same top cluster every time; 1 = top cluster spread evenly across clusters."""
    p = pd.Series(x).value_counts(normalize=True).values
    if len(p) <= 1:
        return 0.0
    return float(-(p * np.log(p)).sum() / np.log(len(p)))


# Baseline real-history views for comparison.
real_view_tbl = cluster_view_table(rets_d_full, clusters)
print("Real-history AE -> BL cluster views:")
display(real_view_tbl[["Members", "LongRunAnn", "RecentAnn", "GapAnn", "GapT", "QAnn", "Coherence"]].round(3))


In [ ]:
# --- Monte Carlo fragility test for the view engine itself ---

SYNTH_N_PER_SCENARIO = 40       # Increase to 200+ for a slower but smoother study.
SYNTH_DAYS = 3 * TRADING_DAYS   # 3 synthetic years per path.
SYNTH_BLOCK = 21                # Roughly one trading month.

scenario_names = list(SYNTH_SCENARIOS.keys()) + ["regime_switching"]

real_signs = real_view_tbl["GapSign"].rename("RealGapSign")
stress_rows = []
detail_rows = []

for scenario in scenario_names:
    for j in range(SYNTH_N_PER_SCENARIO):
        seed_j = SEED + 10_000 * (scenario_names.index(scenario) + 1) + j
        synth_d = make_synthetic_path(
            rets_d_full, clusters, scenario=scenario,
            n_days=SYNTH_DAYS, block_len=SYNTH_BLOCK, seed=seed_j
        )

        tbl = cluster_view_table(synth_d, clusters)
        if tbl.empty:
            continue

        top_cluster = tbl.index[0]
        joined = tbl[["GapSign"]].join(real_signs, how="inner")
        mask = joined["RealGapSign"] != 0
        sign_flip_rate = (
            (joined.loc[mask, "GapSign"] != joined.loc[mask, "RealGapSign"]).mean()
            if mask.any() else np.nan
        )

        ew_synth = synth_d.mean(axis=1)
        stress_rows.append({
            "Scenario": scenario,
            "Path": j,
            "TopViewCluster": top_cluster,
            "TopViewQAnn": tbl.iloc[0]["QAnn"],
            "TopViewGapAnn": tbl.iloc[0]["GapAnn"],
            "MeanAbsGap": tbl["AbsGap"].mean(),
            "MaxAbsGap": tbl["AbsGap"].max(),
            "MeanAbsT": tbl["GapT"].abs().mean(),
            "FragileShareAbsT_lt_1": (tbl["GapT"].abs() < 1.0).mean(),
            "SignFlipRateVsReal": sign_flip_rate,
            "EWAnnReturn": ann_return(ew_synth),
            "EWAnnVol": ann_vol(ew_synth),
            "EWMaxDD": max_drawdown(ew_synth),
        })

        tmp = tbl.reset_index()
        tmp["Scenario"] = scenario
        tmp["Path"] = j
        detail_rows.append(tmp)

view_stress = pd.DataFrame(stress_rows)
view_detail = pd.concat(detail_rows, ignore_index=True)

view_stress_summary = (
    view_stress
    .groupby("Scenario")
    .agg(
        Paths=("Path", "count"),
        TopClusterEntropy=("TopViewCluster", normalized_entropy),
        MedianSignFlip=("SignFlipRateVsReal", "median"),
        P90SignFlip=("SignFlipRateVsReal", lambda x: np.nanquantile(x, 0.90)),
        MedianMeanAbsGap=("MeanAbsGap", "median"),
        MedianMeanAbsT=("MeanAbsT", "median"),
        MedianFragileShare=("FragileShareAbsT_lt_1", "median"),
        MedianEWAnnVol=("EWAnnVol", "median"),
        MedianEWMaxDD=("EWMaxDD", "median"),
    )
    .sort_values("MedianSignFlip", ascending=False)
)

print("View-stability summary across synthetic alternate histories")
display(view_stress_summary.round(3))

top_freq = pd.crosstab(
    view_stress["Scenario"], view_stress["TopViewCluster"],
    normalize="index"
).reindex(scenario_names)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(top_freq.fillna(0).values, aspect="auto")
ax.set_xticks(range(top_freq.shape[1]))
ax.set_xticklabels(top_freq.columns)
ax.set_yticks(range(top_freq.shape[0]))
ax.set_yticklabels(top_freq.index)
ax.set_title("Which latent cluster becomes the dominant AE -> BL view?")
ax.set_xlabel("Top-view latent cluster")
ax.set_ylabel("Synthetic scenario")
plt.colorbar(im, ax=ax, label="Frequency")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
view_stress.boxplot(column="SignFlipRateVsReal", by="Scenario", ax=axes[0], rot=35)
axes[0].set_title("Sign flips vs real-history cluster views")
axes[0].set_ylabel("Share of clusters with flipped gap sign")

view_stress.boxplot(column="MeanAbsT", by="Scenario", ax=axes[1], rot=35)
axes[1].set_title("Mean absolute signal-to-noise of view gaps")
axes[1].set_ylabel("Mean |Gap t-stat|")
plt.suptitle("")
plt.tight_layout()
plt.show()

print("""
How to read this:
- High TopClusterEntropy means the 'most important' view jumps across clusters.
- High SignFlipRate means the mean-reversion direction is regime-dependent.
- Low MeanAbsT means the view is mostly sampling noise, even when QAnn looks large.
A robust research signal should not require these diagnostics to be heroic.
""")


In [ ]:
# --- Does synthetic view instability translate into unstable BL portfolios? ---
# Uses a fixed risk-aversion mean-variance solve for speed.
# The main Section 7 comparison still uses max-Sharpe for BL + AE views.

SYNTH_WEIGHT_PATHS_PER_SCENARIO = 8
BL_SYNTH_RISK_AVERSION = 8.0

w_real_bl = PORT_W["BL + AE views"].reindex(TICKERS).values
w_lw_core = PORT_W["Ledoit-Wolf"].reindex(TICKERS).values

weight_rows = []

for scenario in scenario_names:
    for j in range(SYNTH_WEIGHT_PATHS_PER_SCENARIO):
        seed_j = SEED + 50_000 * (scenario_names.index(scenario) + 1) + j
        synth_d = make_synthetic_path(
            rets_d_full, clusters, scenario=scenario,
            n_days=SYNTH_DAYS, block_len=SYNTH_BLOCK, seed=seed_j
        )
        synth_m = (1 + synth_d).resample("M").prod() - 1
        synth_m = synth_m.dropna(how="any")

        try:
            P_syn, Q_syn, conf_syn = ae_views_for_bl(
                synth_d, clusters, lookback_recent=min(126, len(synth_d) // 2), max_views=3
            )
            if len(Q_syn) == 0:
                raise ValueError("No usable synthetic views.")
            mu_syn_bl, Sig_syn_bl = black_litterman(
                synth_m, mcaps.reindex(TICKERS), P=P_syn, Q=Q_syn, view_conf=conf_syn
            )

            w_syn = mean_variance(
                mu_syn_bl, Sig_syn_bl,
                risk_aversion=BL_SYNTH_RISK_AVERSION,
                max_w=0.20
            )

            w_s = pd.Series(w_syn, index=TICKERS)
            cluster_exp = w_s.groupby(clusters).sum().sort_values(ascending=False)

            weight_rows.append({
                "Scenario": scenario,
                "Path": j,
                "L1FromRealBL": float(np.abs(w_syn - w_real_bl).sum()),
                "L1FromLWCore": float(np.abs(w_syn - w_lw_core).sum()),
                "HHI": herfindahl(w_syn),
                "MaxNameWeight": float(w_s.max()),
                "TopName": w_s.idxmax(),
                "MaxClusterWeight": float(cluster_exp.iloc[0]),
                "TopCluster": cluster_exp.index[0],
                "PosteriorMuDispersion": float(np.std(mu_syn_bl)),
            })
        except Exception as e:
            weight_rows.append({
                "Scenario": scenario,
                "Path": j,
                "Error": str(e)[:120],
            })

weight_fragility = pd.DataFrame(weight_rows)

if "L1FromRealBL" not in weight_fragility.columns:
    print("No synthetic BL portfolios solved. Inspect weight_fragility['Error'].")
    display(weight_fragility.head())
    weight_ok = pd.DataFrame()
else:
    weight_ok = weight_fragility.dropna(subset=["L1FromRealBL"]).copy()

if len(weight_ok) == 0:
    print("No synthetic BL portfolios solved. Inspect weight_fragility['Error'].")
    display(weight_fragility.head())
else:
    weight_summary = (
        weight_ok.groupby("Scenario")
        .agg(
            Paths=("Path", "count"),
            MedianL1FromRealBL=("L1FromRealBL", "median"),
            P90L1FromRealBL=("L1FromRealBL", lambda x: np.quantile(x, 0.90)),
            MedianL1FromLWCore=("L1FromLWCore", "median"),
            MedianHHI=("HHI", "median"),
            MedianMaxClusterWeight=("MaxClusterWeight", "median"),
            TopNameEntropy=("TopName", normalized_entropy),
        )
        .sort_values("MedianL1FromRealBL", ascending=False)
    )

    print("BL posterior-weight fragility under synthetic views")
    display(weight_summary.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    weight_ok.boxplot(column="L1FromRealBL", by="Scenario", ax=axes[0], rot=35)
    axes[0].set_title("Distance from real-history BL portfolio")
    axes[0].set_ylabel("L1 weight distance")

    weight_ok.boxplot(column="MaxClusterWeight", by="Scenario", ax=axes[1], rot=35)
    axes[1].set_title("Latent-cluster concentration under synthetic BL")
    axes[1].set_ylabel("Largest cluster weight")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()

    print("""
Practical reading:
- L1 distance near 0.0 means the synthetic views lead to similar weights.
- L1 distance near 2.0 means almost a complete portfolio reshuffle.
- If synthetic BL frequently moves far from the Ledoit-Wolf core, the view engine
  needs much more OOS evidence before it deserves real capital.
""")


### Synthetic stress-testing summary for the calibrated view engine

The earlier synthetic section tests basic AE -> BL views. This extension checks the calibrated-confidence version and the RMT-backed Black-Litterman portfolio. Synthetic data is still adversarial QA, not proof of alpha.

In [ ]:
# 6) Synthetic stress test of calibrated views + RMT-backed BL weights
CAL_SYNTH_N_PER_SCENARIO = 10
cal_synth_rows = []
cal_synth_weight_rows = []

base_cal_signs = calibrated_views['GapSign'].rename('BaseGapSign') if len(calibrated_views) else pd.Series(dtype=float)
w_cal_real = PORT_W['BL calibrated AE + RMT'].reindex(TICKERS).values if 'BL calibrated AE + RMT' in PORT_W else None
w_lw_core = PORT_W['Ledoit-Wolf'].reindex(TICKERS).values

for scenario in scenario_names:
    for j in range(CAL_SYNTH_N_PER_SCENARIO):
        seed_j = SEED + 90_000 * (scenario_names.index(scenario) + 1) + j
        synth_d = make_synthetic_path(
            rets_d_full, clusters, scenario=scenario,
            n_days=SYNTH_DAYS, block_len=SYNTH_BLOCK, seed=seed_j
        )
        synth_m = (1 + synth_d).resample('M').prod() - 1
        synth_m = synth_m.dropna(how='any')

        P_s, Q_s, conf_s, tbl_s = calibrated_ae_views_for_bl(
            synth_d, clusters, rec_error=None,
            lookback_recent=min(126, len(synth_d) // 2), max_views=5
        )
        if tbl_s.empty:
            continue

        joined = tbl_s[['GapSign']].join(base_cal_signs, how='inner')
        mask = joined['BaseGapSign'] != 0
        sign_flip = float((joined.loc[mask, 'GapSign'] != joined.loc[mask, 'BaseGapSign']).mean()) if mask.any() else np.nan

        cal_synth_rows.append({
            'Scenario': scenario,
            'Path': j,
            'TopCluster': tbl_s.index[0],
            'MedianConfidence': tbl_s['Confidence'].median(),
            'MedianFragility': tbl_s['Fragility'].median(),
            'MedianGapSNR': tbl_s['GapSNR'].median(),
            'SignFlipRateVsBase': sign_flip,
        })

        try:
            Sig_s_rmt = rmt_denoised_cov(synth_m, periods=12)
            mu_s_bl, Sig_s_bl = black_litterman_given_sigma(
                Sig_s_rmt, mcaps.reindex(TICKERS), P=P_s, Q=Q_s, view_conf=conf_s
            )
            w_s = mean_variance(mu_s_bl, Sig_s_bl, risk_aversion=BL_SYNTH_RISK_AVERSION, max_w=0.20)
            ws = pd.Series(w_s, index=TICKERS)
            cluster_exp = ws.groupby(clusters).sum().sort_values(ascending=False)
            cal_synth_weight_rows.append({
                'Scenario': scenario,
                'Path': j,
                'L1FromRealCalBL': float(np.abs(w_s - w_cal_real).sum()) if w_cal_real is not None else np.nan,
                'L1FromLWCore': float(np.abs(w_s - w_lw_core).sum()),
                'HHI': herfindahl(w_s),
                'TopName': ws.idxmax(),
                'MaxClusterWeight': float(cluster_exp.iloc[0]),
                'TopCluster': cluster_exp.index[0],
            })
        except Exception as e:
            cal_synth_weight_rows.append({'Scenario': scenario, 'Path': j, 'Error': str(e)[:120]})

calibrated_synth_detail = pd.DataFrame(cal_synth_rows)
calibrated_weight_fragility = pd.DataFrame(cal_synth_weight_rows)

if len(calibrated_synth_detail):
    calibrated_synth_summary = calibrated_synth_detail.groupby('Scenario').agg(
        Paths=('Path', 'count'),
        TopClusterEntropy=('TopCluster', normalized_entropy),
        MedianConfidence=('MedianConfidence', 'median'),
        MedianFragility=('MedianFragility', 'median'),
        MedianGapSNR=('MedianGapSNR', 'median'),
        MedianSignFlip=('SignFlipRateVsBase', 'median'),
        P90SignFlip=('SignFlipRateVsBase', lambda x: np.nanquantile(x, 0.90)),
    ).sort_values('MedianFragility', ascending=False)
    print('Calibrated view-engine fragility under synthetic regimes:')
    display(calibrated_synth_summary.round(3))

if len(calibrated_weight_fragility) and 'L1FromLWCore' in calibrated_weight_fragility:
    calibrated_weight_ok = calibrated_weight_fragility.dropna(subset=['L1FromLWCore'])
    calibrated_weight_summary = calibrated_weight_ok.groupby('Scenario').agg(
        Paths=('Path', 'count'),
        MedianL1FromRealCalBL=('L1FromRealCalBL', 'median'),
        MedianL1FromLWCore=('L1FromLWCore', 'median'),
        P90L1FromLWCore=('L1FromLWCore', lambda x: np.nanquantile(x, 0.90)),
        MedianHHI=('HHI', 'median'),
        MedianMaxClusterWeight=('MaxClusterWeight', 'median'),
        TopNameEntropy=('TopName', normalized_entropy),
    ).sort_values('MedianL1FromLWCore', ascending=False)
    print('Calibrated RMT-BL portfolio fragility under synthetic regimes:')
    display(calibrated_weight_summary.round(3))

## 10. OOS uncertainty: how much should we believe the walk-forward ranking?

This is the other half of Iñaki's feedback. A short out-of-sample window can produce an attractive ranking that is mostly sampling noise.

The next cell adds two conservative checks:

1. **Approximate Sharpe confidence intervals** using the length of the OOS period.
2. **Paired block-bootstrap tests** comparing the fancy models with Ledoit-Wolf on the same resampled dates.

The target is not to find a p-value that "approves" the strategy. It is to make the uncertainty visible enough that we do not over-interpret a lucky ranking.


In [ ]:
# --- OOS sample-size and ranking uncertainty diagnostics ---

def sharpe_se_approx(r, periods=TRADING_DAYS):
    """
    Rough IID standard error for an annualized Sharpe over T years.
    This is optimistic for financial returns because serial dependence,
    regime clustering, and non-normality usually make uncertainty larger.
    """
    r = pd.Series(r).dropna()
    years = len(r) / periods
    s = sharpe(r, periods=periods)
    if years <= 0 or not np.isfinite(s):
        return np.nan
    return np.sqrt((1 + 0.5 * s ** 2) / years)


oos_rows = {}
for name, r in bt.items():
    rr = pd.Series(r).dropna()
    yrs = len(rr) / TRADING_DAYS
    s = sharpe(rr)
    se = sharpe_se_approx(rr)
    oos_rows[name] = {
        "Days": len(rr),
        "Years": yrs,
        "Sharpe": s,
        "ApproxSharpeSE": se,
        "SharpeCI_low": s - 1.96 * se if np.isfinite(se) else np.nan,
        "SharpeCI_high": s + 1.96 * se if np.isfinite(se) else np.nan,
        "AnnReturn": ann_return(rr),
        "MaxDD": max_drawdown(rr),
    }

oos_uncertainty = pd.DataFrame(oos_rows).T.sort_values("Sharpe", ascending=False)
print("OOS Sharpe uncertainty — wide intervals mean rankings are fragile")
display(oos_uncertainty.round(3))


def paired_block_bootstrap_diffs(challenger, benchmark, n_boot=1000,
                                 block_len=21, seed=SEED):
    """
    Paired bootstrap: both strategies are resampled on the same dates.
    This keeps common market regimes aligned and estimates uncertainty in the
    *difference* between strategies, not just each strategy separately.
    """
    df = pd.concat(
        [pd.Series(challenger).rename("challenger"),
         pd.Series(benchmark).rename("benchmark")],
        axis=1,
        join="inner"
    ).dropna()

    n = len(df)
    if n <= block_len:
        raise ValueError("Not enough OOS observations for the selected block length.")

    rng = np.random.default_rng(seed)
    sharpe_diffs, wealth_diffs = [], []

    for _ in range(n_boot):
        starts = rng.integers(0, n - block_len + 1,
                              size=int(np.ceil(n / block_len)))
        idx = np.concatenate([np.arange(s, s + block_len) for s in starts])[:n]

        ch = pd.Series(df["challenger"].values[idx])
        bm = pd.Series(df["benchmark"].values[idx])

        sharpe_diffs.append(sharpe(ch) - sharpe(bm))
        wealth_diffs.append((1 + ch).prod() - (1 + bm).prod())

    return pd.DataFrame({
        "SharpeDiff": sharpe_diffs,
        "TerminalWealthDiff_per_1": wealth_diffs,
    })


BOOT_N = 1000
BOOT_BLOCK = 21
pairs = [
    ("BL calibrated AE + RMT", "Ledoit-Wolf"),
    ("BL + AE views", "Ledoit-Wolf"),
    ("RMT denoised", "Ledoit-Wolf"),
    ("AE factor", "Ledoit-Wolf"),
    ("DAE denoised", "Ledoit-Wolf"),
    ("PCA factor", "Ledoit-Wolf"),
]

boot_summaries = []
boot_diffs_for_plot = {}

for challenger, benchmark in pairs:
    if challenger not in bt or benchmark not in bt:
        continue

    diffs = paired_block_bootstrap_diffs(
        bt[challenger], bt[benchmark],
        n_boot=BOOT_N, block_len=BOOT_BLOCK,
        seed=SEED + len(boot_summaries)
    )

    boot_diffs_for_plot[f"{challenger} - {benchmark}"] = diffs["SharpeDiff"].values
    boot_summaries.append({
        "Comparison": f"{challenger} - {benchmark}",
        "MedianSharpeDiff": diffs["SharpeDiff"].median(),
        "SharpeDiff_5pct": diffs["SharpeDiff"].quantile(0.05),
        "SharpeDiff_95pct": diffs["SharpeDiff"].quantile(0.95),
        "P(SharpeDiff > 0)": (diffs["SharpeDiff"] > 0).mean(),
        "MedianTerminalWealthDiff_per_1000EUR": 1000 * diffs["TerminalWealthDiff_per_1"].median(),
    })

boot_summary = pd.DataFrame(boot_summaries).set_index("Comparison")
print("Paired block-bootstrap uncertainty vs Ledoit-Wolf")
display(boot_summary.round(3))

fig, ax = plt.subplots(figsize=(11, 4))
pd.DataFrame(boot_diffs_for_plot).boxplot(ax=ax, rot=25)
ax.axhline(0, lw=1)
ax.set_title("Bootstrap distribution of Sharpe differences vs Ledoit-Wolf")
ax.set_ylabel("Sharpe difference")
plt.tight_layout()
plt.show()

print("""
Decision rule:
If the 5%-95% Sharpe-difference interval crosses zero, the OOS ranking is not
statistically stable enough to be treated as evidence of deployable alpha.
That does not make the research useless; it means the correct next step is
frozen-code paper trading and more OOS accumulation, not immediate confidence.
""")


### OOS uncertainty decision flags

The backtest ranking should be converted into a decision table. A strategy can look good while still failing the uncertainty test if the paired bootstrap interval crosses zero or the OOS sample is too short.

In [ ]:
# 7) OOS uncertainty converted into decision flags

def build_oos_decision_table(oos_uncertainty, boot_summary, min_oos_years=3.0):
    rows = []
    for name, row in oos_uncertainty.iterrows():
        years = row.get('Years', np.nan)
        sharpe_low = row.get('SharpeCI_low', np.nan)
        sharpe_high = row.get('SharpeCI_high', np.nan)
        sample_flag = 'OK' if years >= min_oos_years else 'TOO_SHORT'
        sharpe_flag = 'POSITIVE_CI' if sharpe_low > 0 else 'CROSSES_OR_BELOW_ZERO'
        rows.append({
            'Strategy': name,
            'OOSYears': years,
            'Sharpe': row.get('Sharpe', np.nan),
            'SharpeCI_low': sharpe_low,
            'SharpeCI_high': sharpe_high,
            'SampleFlag': sample_flag,
            'SharpeFlag': sharpe_flag,
        })
    out = pd.DataFrame(rows).set_index('Strategy')

    if boot_summary is not None and len(boot_summary):
        for comp, brow in boot_summary.iterrows():
            challenger = comp.split(' - ')[0]
            crosses_zero = brow['SharpeDiff_5pct'] <= 0 <= brow['SharpeDiff_95pct']
            if challenger in out.index:
                out.loc[challenger, 'BootstrapVsLW'] = 'CROSSES_ZERO' if crosses_zero else 'ABOVE_ZERO'
                out.loc[challenger, 'P_SharpeDiff_gt_0'] = brow['P(SharpeDiff > 0)']
    out['DeploymentEvidence'] = np.where(
        (out['SampleFlag'] == 'OK') &
        (out['SharpeFlag'] == 'POSITIVE_CI') &
        (out.get('BootstrapVsLW', pd.Series(index=out.index, dtype=object)).fillna('NA') == 'ABOVE_ZERO'),
        'STRONGER_EVIDENCE',
        'RESEARCH_ONLY_OR_PAPER_TRADE'
    )
    return out


oos_decision_table = build_oos_decision_table(oos_uncertainty, boot_summary)
print('OOS decision flags. These flags intentionally err on the conservative side.')
display(oos_decision_table.round(3))

### Final stability dashboard

This is the research-governance view. It summarizes whether the project is producing a stable process or only an attractive backtest. The dashboard is deliberately conservative: a model can be intellectually interesting and still fail the deployment-evidence test.

In [ ]:
# 8) Final stability dashboard

def status_watch(value, good_condition, warn_condition=True):
    if pd.isna(value):
        return 'NA'
    if good_condition(value):
        return 'PASS'
    if warn_condition is True or warn_condition(value):
        return 'WATCH'
    return 'FAIL'


dashboard_rows = []

pca_ae_ratio = ae_mse / pca_mse if np.isfinite(pca_mse) and pca_mse > 0 else np.nan
dashboard_rows.append({
    'Area': 'Autoencoder',
    'Metric': 'AE validation MSE / PCA validation MSE',
    'Value': pca_ae_ratio,
    'Status': status_watch(pca_ae_ratio, lambda x: x < 0.98, lambda x: x <= 1.05),
    'Interpretation': 'AE must beat a simple linear baseline to claim reconstruction value.'
})

avg_conf = calibrated_views['Confidence'].mean() if len(calibrated_views) else np.nan
med_flip = calibrated_views['FlipRate'].median() if len(calibrated_views) else np.nan
dashboard_rows.append({
    'Area': 'Views',
    'Metric': 'Average calibrated view confidence',
    'Value': avg_conf,
    'Status': status_watch(avg_conf, lambda x: x >= 0.55, lambda x: x >= 0.35),
    'Interpretation': 'Low confidence means BL views should be weak tilts, not core positions.'
})
dashboard_rows.append({
    'Area': 'Views',
    'Metric': 'Median rolling view sign-flip rate',
    'Value': med_flip,
    'Status': status_watch(med_flip, lambda x: x <= 0.25, lambda x: x <= 0.50),
    'Interpretation': 'Frequent sign flips mean the view direction is regime fragile.'
})

noise_share = rmt_summary['Noise_eigenvalue_share'] if 'rmt_summary' in globals() else np.nan
dashboard_rows.append({
    'Area': 'Risk model',
    'Metric': 'RMT noise eigenvalue share',
    'Value': noise_share,
    'Status': status_watch(noise_share, lambda x: 0.20 <= x <= 0.95, lambda x: True),
    'Interpretation': 'High noise share supports using shrinkage/RMT rather than raw sample covariance.'
})

if 'BL calibrated AE + RMT' in PORT_W:
    l1_cal_lw = float(np.abs(PORT_W['BL calibrated AE + RMT'].values - PORT_W['Ledoit-Wolf'].values).sum())
    net_cal = network_concentration_score(PORT_W['BL calibrated AE + RMT'].values) if 'network_concentration_score' in globals() else np.nan
else:
    l1_cal_lw, net_cal = np.nan, np.nan

dashboard_rows.append({
    'Area': 'Allocation',
    'Metric': 'L1 distance: calibrated BL+RMT vs Ledoit-Wolf',
    'Value': l1_cal_lw,
    'Status': status_watch(l1_cal_lw, lambda x: x <= 0.50, lambda x: x <= 1.00),
    'Interpretation': 'Large distance from the robust core means the views are dominating the portfolio.'
})
dashboard_rows.append({
    'Area': 'Network',
    'Metric': 'Network concentration of calibrated BL+RMT',
    'Value': net_cal,
    'Status': status_watch(net_cal, lambda x: x <= 1.05, lambda x: x <= 1.25),
    'Interpretation': 'Above 1 means the portfolio tilts toward central, crowded names.'
})

if 'view_stress_summary' in globals() and len(view_stress_summary):
    worst_basic_flip = view_stress_summary['P90SignFlip'].max()
else:
    worst_basic_flip = np.nan
dashboard_rows.append({
    'Area': 'Synthetic stress',
    'Metric': 'Worst scenario P90 sign-flip rate, basic views',
    'Value': worst_basic_flip,
    'Status': status_watch(worst_basic_flip, lambda x: x <= 0.35, lambda x: x <= 0.60),
    'Interpretation': 'Synthetic sign flips reveal whether view direction depends heavily on regime assumptions.'
})

if 'calibrated_weight_summary' in globals() and len(calibrated_weight_summary):
    worst_cal_l1 = calibrated_weight_summary['P90L1FromLWCore'].max()
else:
    worst_cal_l1 = np.nan
dashboard_rows.append({
    'Area': 'Synthetic stress',
    'Metric': 'Worst scenario P90 L1 from LW core, calibrated BL+RMT',
    'Value': worst_cal_l1,
    'Status': status_watch(worst_cal_l1, lambda x: x <= 0.75, lambda x: x <= 1.25),
    'Interpretation': 'Large reshuffles under synthetic paths mean the view engine is fragile.'
})

if 'BL calibrated AE + RMT' in oos_decision_table.index:
    oos_years_cal = oos_decision_table.loc['BL calibrated AE + RMT', 'OOSYears']
    deploy_flag_cal = oos_decision_table.loc['BL calibrated AE + RMT', 'DeploymentEvidence']
elif 'BL + AE views' in oos_decision_table.index:
    oos_years_cal = oos_decision_table.loc['BL + AE views', 'OOSYears']
    deploy_flag_cal = oos_decision_table.loc['BL + AE views', 'DeploymentEvidence']
else:
    oos_years_cal, deploy_flag_cal = np.nan, 'NA'

dashboard_rows.append({
    'Area': 'OOS evidence',
    'Metric': 'OOS years for main BL overlay',
    'Value': oos_years_cal,
    'Status': status_watch(oos_years_cal, lambda x: x >= 5.0, lambda x: x >= 3.0),
    'Interpretation': 'Short OOS samples should not support deployment confidence.'
})


if 'true_holdout_test' in globals() and 'true_holdout_primary_model' in globals():
    try:
        holdout_sharpe = float(true_holdout_test.loc[true_holdout_primary_model, 'Sharpe'])
        holdout_years = len(true_holdout_test_returns[true_holdout_primary_model].dropna()) / TRADING_DAYS
        holdout_flag = 'FINAL_TEST_POSITIVE' if holdout_sharpe > 0 else 'FINAL_TEST_WEAK'
    except Exception:
        holdout_sharpe, holdout_years, holdout_flag = np.nan, np.nan, 'NA'
else:
    holdout_sharpe, holdout_years, holdout_flag = np.nan, np.nan, 'NOT_RUN'

dashboard_rows.append({
    'Area': 'True hold-out',
    'Metric': 'Validation-selected model Sharpe on 2023-present test',
    'Value': holdout_sharpe,
    'Status': status_watch(holdout_sharpe, lambda x: x > 0.5 and holdout_years >= 2.0, lambda x: x > 0.0),
    'Interpretation': 'This is the clean final-test evidence; do not use it to tune the model.'
})

dashboard_rows.append({
    'Area': 'OOS evidence',
    'Metric': 'Deployment evidence flag',
    'Value': deploy_flag_cal,
    'Status': 'PASS' if deploy_flag_cal == 'STRONGER_EVIDENCE' else 'WATCH',
    'Interpretation': 'Even a good model should stay research-only if bootstrap evidence is weak.'
})

stability_dashboard = pd.DataFrame(dashboard_rows)
print('Final stability dashboard')
display(stability_dashboard)

watch_count = (stability_dashboard['Status'] == 'WATCH').sum()
fail_count = (stability_dashboard['Status'] == 'FAIL').sum()
print(f'Conservative decision: {watch_count} WATCH flags, {fail_count} FAIL flags.')
print('Recommended interpretation: use this as a research and paper-trading framework until frozen-code OOS evidence becomes materially larger.')

### Research governance: how I would move this from notebook to decision process

A clean next version would separate **research** from **evidence**:

- **Freeze the view engine**: no architecture, ticker-universe, cluster-count, or hyperparameter changes after the research cutoff.
- **Paper trade the forecasts**: log every BL view, posterior expected return, weight vector, and benchmark weight before returns are known.
- **Pre-register kill criteria**: stop trusting the AE → BL overlay if sign flips, cluster concentration, turnover, or drawdown exceed pre-defined thresholds.
- **Treat synthetic tests as adversarial QA**: they are excellent for finding fragility and poor for proving alpha.
- **Delay deployment confidence** until the frozen process has accumulated a materially larger OOS sample across more regimes.


## 11. Visualizations

Every required chart appears inline next to the analysis it supports — price history & correlations (§2), explained variance & loadings (§4), training curves, latent space, clusters, anomaly regimes, VAE latent/QQ (§5), EVT tail (§6), allocations & frontier (§3, §7), cumulative returns, drawdowns, rolling vol & Sharpe (§8), synthetic stress tests (§9), OOS uncertainty diagnostics (§10), and the final stability dashboard. Co-locating charts with analysis beats a graveyard "plots section" — you should never see a chart without the argument it belongs to.

## 12. Interpretation — read this part slowly

**Which portfolio for a long-term investor?**
The defensible answer is boring: **Ledoit–Wolf minimum-variance or HRP as the core**, with the position cap doing more work than the optimizer. Across decades of literature, shrinkage + simple structure beats clever `mu` forecasting out-of-sample. If the AE-factor portfolio ranked above LW in the walk-forward, note the sample is one short, regime-poor window — that ranking would need to survive different periods, universes and seeds before you'd trust it. **And at €1,000 specifically, the genuinely optimal portfolio is one accumulating world/US ETF until the account is large enough (≳€20–50k) for fixed costs and complexity to be worth it.**



**What Iñaki's synthetic-data point added:** synthetic histories are now used as an adversarial QA layer for the AE → BL view engine. The right interpretation is asymmetric: if views flip signs, top clusters rotate randomly, or BL weights move far away from the Ledoit-Wolf core under mild perturbations, that is strong negative evidence. But even stable synthetic behavior is not positive proof of alpha, because it is still generated from assumptions learned from the same limited history.

**What Iñaki's OOS warning changed:** the walk-forward table is now treated as a ranking hypothesis, not a deployment conclusion. The bootstrap and Sharpe-interval diagnostics make the uncertainty explicit. If the confidence bands around model differences cross zero, the intellectually honest conclusion is "promising research process, insufficient live evidence," not "ready to allocate with confidence."

**Did autoencoder factors improve diversification?**
Where AEs earned their keep here was **not** reconstruction (PCA is near-unbeatable on 18 assets dominated by one linear market factor) but **structure discovery**: latent clustering exposed that the "8 themes" collapse into ~4–5 real risk groups, and reconstruction-error spikes located stress regimes without being told where they were. That's a diversification *audit* tool more than a return enhancer.

**Hidden concentration via latent clustering — the practical takeaway:**
Cap exposure per *latent cluster*, not per ticker or per labeled sector. A 20%-per-name cap still allows ~80% in one latent cluster if four cluster-mates each get 20%.

**When does ML overfit financial data?** Almost by default, because:
- Signal-to-noise in returns is brutally low (daily R² of any predictor is tiny)
- Markets are non-stationary — the "function" being learned changes underneath you
- One historical path: you cannot cross-validate across alternate histories
- Researcher degrees of freedom: every architecture/seed/hyperparameter retry is a silent multiple-comparison
Defenses used here: chronological splits, early stopping, tiny networks, walk-forward-only conclusions, and — most important — *willingness to report that the fancy model didn't win*.

**Why this is not financial advice:** hindsight-selected universe, one short sample, no taxes (relevant for a Madrid-based investor: Spanish CGT 19–28%), no personal-circumstance inputs, parameter sensitivity everywhere, and models that assume the future resembles a past that included the greatest large-cap bull run in history.

---
### Final recommended allocation for €1,000 (educational portfolio)
Built next cell: **Ledoit–Wolf min-variance ∩ latent-cluster caps**, rounded to fractional shares. Plus the honest alternative printed alongside it.

**OOS discipline update:** the walk-forward backtest is a rolling diagnostic, not the final truth. The stricter evidence comes from the chronological hold-out section: train on pre-2020, validate on 2020-2022, and evaluate 2023-present only after the model is frozen. Because the full 18-name universe does not have enough pre-2020 history, the notebook explicitly flags the strict all-18 hold-out as infeasible and runs the clean test on the largest eligible sub-universe instead.


In [ ]:
# Final allocation: LW min-variance with a 35% cap per LATENT CLUSTER
def min_variance_cluster_capped(Sigma, clusters, max_w=0.20, cluster_cap=0.35):
    n = Sigma.shape[0]; w = cp.Variable(n)
    cons = [cp.sum(w) == 1, w >= 0, w <= max_w]
    for c in sorted(clusters.unique()):
        idx = [TICKERS.index(t) for t in clusters[clusters == c].index]
        cons.append(cp.sum(w[idx]) <= cluster_cap)
    return solve_qp(cp.Minimize(cp.quad_form(w, cp.psd_wrap(Sigma))), cons, w)

w_final = min_variance_cluster_capped(lw_sigma(rets_m_full), clusters)
final = pd.DataFrame({
    "Weight": w_final,
    "EUR": np.round(w_final * INITIAL_CAPITAL_EUR, 2),
    "Theme": [UNIVERSE[t] for t in TICKERS],
    "LatentCluster": clusters.values,
}, index=TICKERS).sort_values("Weight", ascending=False)
final = final[final["Weight"] > 0.005]

print("FINAL EDUCATIONAL PORTFOLIO — €1,000, LW min-variance + latent-cluster caps")
print(final.round(3).to_string())
print(f"\nHHI: {herfindahl(w_final):.3f} | names held: {len(final)}")
print("\nSector exposure:"); print(sector_exposure(w_final).round(2).to_string())
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(final["Weight"], labels=final.index, autopct="%1.0f%%", pctdistance=0.8)
ax.set_title("Final allocation (€1,000)"); plt.show()

print("""
THE HONEST ALTERNATIVE
At €1,000, with fixed brokerage minimums and your time valued at anything > €0/hr,
a single accumulating broad ETF (e.g., an MSCI World / S&P 500 UCITS accumulator)
dominates this 15-name portfolio on costs, taxes, and behavioral robustness.
This notebook's value is the METHODOLOGY — which becomes genuinely decision-relevant
somewhere north of €20–50k, and professionally relevant at any AUM.
Not financial advice.""")


## 13. Final discovery visuals: 3D maps for the research story

These are the most postable charts in the notebook because they visualize the actual research claim instead of just showing a backtest.

**Figure A — Market-state discovery map**  
Each point is a trading day. The three axes are compressed autoencoder latent factors. Color shows the detected regime, and marker size shows reconstruction-error stress. If stress periods form visible pockets or tails, the autoencoder is capturing regime structure rather than only reconstructing noise.

**Figure B — Asset-cluster discovery map**  
Each point is one stock. The axes are the asset's exposure to the autoencoder latent factors. Color shows latent cluster membership, and marker size shows final portfolio weight. This makes hidden concentration visible: if large positions sit in the same latent cloud, the portfolio is less diversified than the ticker count suggests.

These visuals are **not proof of alpha**. They are discovery and governance visuals: they show whether the model found stable structure worth stress-testing further.

In [ ]:
# Final Discovery Visual A: 3D autoencoder latent-regime market map
from pathlib import Path
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 - needed by matplotlib for 3D projection
from matplotlib.lines import Line2D

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)


def _safe_zscore_series(x):
    s = pd.Series(x).astype(float)
    std = s.std(ddof=0)
    if not np.isfinite(std) or std == 0:
        std = 1.0
    return (s - s.mean()) / std


def _build_latent_market_frame():
    """Build a plotting frame for dates in 3D latent market-state space."""
    if 'latent_z' in globals():
        L = pd.DataFrame(latent_z).copy()
    elif 'Z_df' in globals():
        L = pd.DataFrame(Z_df).copy()
        L = (L - L.mean()) / L.std(ddof=0)
    elif {'ae', 'scaler', 'rets_d_full'}.issubset(globals()):
        X_tmp = scaler.transform(rets_d_full.values)
        with torch.no_grad():
            Z_tmp = ae.encoder(torch.tensor(X_tmp, dtype=torch.float32, device=DEVICE)).cpu().numpy()
        L = pd.DataFrame(Z_tmp, index=rets_d_full.index, columns=[f'L{i+1}' for i in range(Z_tmp.shape[1])])
        L = (L - L.mean()) / L.std(ddof=0)
    else:
        raise RuntimeError('No latent matrix found. Run the autoencoder section first.')

    L = L.replace([np.inf, -np.inf], np.nan).dropna(how='all').fillna(0.0)

    helper_cols = []
    if 'rets_d_full' in globals():
        ew = rets_d_full.mean(axis=1).reindex(L.index)
        helper_cols.append(_safe_zscore_series(ew.rolling(21).std() * np.sqrt(TRADING_DAYS)).rename('RealizedVol21'))
        helper_cols.append(_safe_zscore_series(ew.rolling(63).mean() * TRADING_DAYS).rename('Momentum63'))
    if 'rec_err' in globals():
        helper_cols.append(_safe_zscore_series(pd.Series(rec_err).reindex(L.index)).rename('ReconErrorZ'))

    if L.shape[1] >= 3:
        coords = L.iloc[:, :3].values
        coord_source = 'raw AE latent factors L1/L2/L3'
    else:
        X_plot = pd.concat([L] + helper_cols, axis=1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        if X_plot.shape[1] >= 3:
            coords = PCA(n_components=3, random_state=SEED).fit_transform(X_plot.values)
            coord_source = 'PCA projection of AE latent factors plus regime helpers'
        elif X_plot.shape[1] == 2:
            coords = np.column_stack([X_plot.values, np.zeros(len(X_plot))])
            coord_source = '2D latent factors with zero third axis'
        else:
            coords = np.column_stack([X_plot.iloc[:, 0].values, np.zeros(len(X_plot)), np.zeros(len(X_plot))])
            coord_source = '1D latent factor with zero helper axes'

    df = pd.DataFrame(coords, index=L.index, columns=['LatentAxis1', 'LatentAxis2', 'LatentAxis3'])
    df.index.name = 'Date'

    if 'regime_labels' in globals():
        df['Regime'] = pd.Series(regime_labels).reindex(df.index).fillna('unknown').astype(str)
    elif 'build_regime_labels' in globals() and 'rets_d_full' in globals():
        df['Regime'] = build_regime_labels(rets_d_full, stress if 'stress' in globals() else None).reindex(df.index).fillna('unknown').astype(str)
    else:
        df['Regime'] = 'unknown'

    if 'rec_err' in globals():
        df['ReconstructionError'] = pd.Series(rec_err).reindex(df.index).astype(float)
    else:
        df['ReconstructionError'] = np.nan

    if df['ReconstructionError'].notna().any():
        stress_rank = df['ReconstructionError'].rank(pct=True).fillna(0.5)
    else:
        stress_rank = pd.Series(0.5, index=df.index)
    df['MarkerSize'] = 12 + 90 * stress_rank

    if 'rets_d_full' in globals():
        ew = rets_d_full.mean(axis=1).reindex(df.index)
        df['EW_Return'] = ew
        df['EW_21dVolAnn'] = ew.rolling(21).std() * np.sqrt(TRADING_DAYS)
        df['EW_63dMomentumAnn'] = ew.rolling(63).mean() * TRADING_DAYS

    return df, coord_source


discovery_market_3d, coord_source = _build_latent_market_frame()
regime_codes, regime_names = pd.factorize(discovery_market_3d['Regime'])

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    discovery_market_3d['LatentAxis1'],
    discovery_market_3d['LatentAxis2'],
    discovery_market_3d['LatentAxis3'],
    c=regime_codes,
    cmap='tab10',
    s=discovery_market_3d['MarkerSize'],
    alpha=0.68,
    linewidths=0
)
ax.set_title('3D Autoencoder Market-State Discovery Map\ncolor = regime | size = reconstruction-error stress', pad=18)
ax.set_xlabel('Latent axis 1')
ax.set_ylabel('Latent axis 2')
ax.set_zlabel('Latent axis 3')

# Label only the most abnormal dates to keep the chart readable.
if discovery_market_3d['ReconstructionError'].notna().any():
    top_stress = discovery_market_3d.nlargest(min(7, len(discovery_market_3d)), 'ReconstructionError')
    for dt, row in top_stress.iterrows():
        ax.text(row['LatentAxis1'], row['LatentAxis2'], row['LatentAxis3'], str(pd.Timestamp(dt).date()), fontsize=7)

cmap = plt.get_cmap('tab10', max(1, len(regime_names)))
legend_handles = [
    Line2D([0], [0], marker='o', color='w', label=str(name),
           markerfacecolor=cmap(i), markersize=8)
    for i, name in enumerate(regime_names)
]
ax.legend(handles=legend_handles, title='Regime', loc='upper left', bbox_to_anchor=(1.02, 0.95))
plt.tight_layout()

market_png = OUTPUT_DIR / 'final_3d_market_state_discovery.png'
plt.savefig(market_png, dpi=220, bbox_inches='tight')
plt.show()

print(f'Saved static market-state discovery image to: {market_png}')
print(f'Coordinate source: {coord_source}')

market_summary = (
    discovery_market_3d
    .groupby('Regime')
    .agg(
        Days=('Regime', 'size'),
        MedianReconError=('ReconstructionError', 'median'),
        Mean21dVolAnn=('EW_21dVolAnn', 'mean') if 'EW_21dVolAnn' in discovery_market_3d else ('MarkerSize', 'mean'),
        Mean63dMomentumAnn=('EW_63dMomentumAnn', 'mean') if 'EW_63dMomentumAnn' in discovery_market_3d else ('MarkerSize', 'mean')
    )
    .sort_values('Days', ascending=False)
)
print('Regime summary behind the 3D visual:')
display(market_summary.round(4))

# Optional interactive version for notebooks that have plotly installed.
try:
    import plotly.express as px
    plotly_frame = discovery_market_3d.reset_index()
    plotly_frame['Date'] = plotly_frame['Date'].astype(str)
    market_html = OUTPUT_DIR / 'interactive_3d_market_state_discovery.html'
    fig_px = px.scatter_3d(
        plotly_frame,
        x='LatentAxis1', y='LatentAxis2', z='LatentAxis3',
        color='Regime',
        size='MarkerSize',
        hover_data=[c for c in ['Date', 'ReconstructionError', 'EW_Return', 'EW_21dVolAnn', 'EW_63dMomentumAnn'] if c in plotly_frame.columns],
        title='Interactive 3D Autoencoder Market-State Discovery Map'
    )
    fig_px.write_html(market_html)
    fig_px.show()
    print(f'Saved interactive market-state discovery map to: {market_html}')
except Exception as e:
    print(f'Interactive Plotly market map skipped: {e}')

In [ ]:
# Final Discovery Visual B: 3D asset-cluster map with portfolio weights

def _build_asset_discovery_frame():
    """Build a plotting frame for assets in 3D latent-exposure space."""
    if 'asset_emb' in globals():
        A = pd.DataFrame(asset_emb).copy()
    elif {'Z_df', 'rets_d_full'}.issubset(globals()):
        # Fallback: asset exposure to each latent market factor.
        A = pd.DataFrame(
            [[pd.Series(rets_d_full[t]).corr(pd.Series(Z_df[f], index=Z_df.index)) for f in Z_df.columns]
             for t in rets_d_full.columns],
            index=rets_d_full.columns,
            columns=Z_df.columns
        )
    else:
        raise RuntimeError('No asset latent exposure matrix found. Run the latent clustering section first.')

    A = A.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if A.shape[1] >= 3:
        coords = A.iloc[:, :3].values
        coord_source = 'raw asset exposures to AE latent factors'
    elif A.shape[1] >= 2:
        coords = np.column_stack([A.iloc[:, :2].values, np.zeros(len(A))])
        coord_source = '2D asset latent exposure map with zero third axis'
    else:
        coords = np.column_stack([A.iloc[:, 0].values, np.zeros(len(A)), np.zeros(len(A))])
        coord_source = '1D asset latent exposure map with zero helper axes'

    df = pd.DataFrame(coords, index=A.index, columns=['LatentExposure1', 'LatentExposure2', 'LatentExposure3'])
    df.index.name = 'Ticker'
    df['Theme'] = [UNIVERSE.get(t, 'Unknown') if 'UNIVERSE' in globals() else 'Unknown' for t in df.index]

    if 'clusters' in globals():
        df['LatentCluster'] = pd.Series(clusters).reindex(df.index).astype('Int64').astype(str)
    else:
        df['LatentCluster'] = 'unknown'

    if 'w_final' in globals():
        weights = pd.Series(w_final, index=TICKERS).reindex(df.index).fillna(0.0)
        weight_source = 'final LW min-variance + latent-cluster caps'
    elif 'PORT_W' in globals() and 'BL calibrated AE + RMT' in PORT_W:
        weights = PORT_W['BL calibrated AE + RMT'].reindex(df.index).fillna(0.0)
        weight_source = 'BL calibrated AE + RMT'
    elif 'PORT_W' in globals() and 'Ledoit-Wolf' in PORT_W:
        weights = PORT_W['Ledoit-Wolf'].reindex(df.index).fillna(0.0)
        weight_source = 'Ledoit-Wolf'
    else:
        weights = pd.Series(1.0 / len(df), index=df.index)
        weight_source = 'equal weight fallback'

    df['Weight'] = weights.astype(float)
    df['EUR_per_1000'] = df['Weight'] * 1000.0
    df['MarkerSize'] = 70 + 900 * df['Weight'].clip(lower=0)
    return df, coord_source, weight_source


asset_discovery_3d, asset_coord_source, asset_weight_source = _build_asset_discovery_frame()
cluster_codes, cluster_names = pd.factorize(asset_discovery_3d['LatentCluster'])

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(
    asset_discovery_3d['LatentExposure1'],
    asset_discovery_3d['LatentExposure2'],
    asset_discovery_3d['LatentExposure3'],
    c=cluster_codes,
    cmap='tab10',
    s=asset_discovery_3d['MarkerSize'],
    alpha=0.78,
    linewidths=0.5
)

for ticker, row in asset_discovery_3d.iterrows():
    ax.text(row['LatentExposure1'], row['LatentExposure2'], row['LatentExposure3'], ticker, fontsize=8)

ax.set_title('3D Asset Discovery Map\ncolor = AE latent cluster | size = final portfolio weight', pad=18)
ax.set_xlabel('Latent exposure 1')
ax.set_ylabel('Latent exposure 2')
ax.set_zlabel('Latent exposure 3')

cmap = plt.get_cmap('tab10', max(1, len(cluster_names)))
legend_handles = [
    Line2D([0], [0], marker='o', color='w', label=f'Cluster {name}',
           markerfacecolor=cmap(i), markersize=8)
    for i, name in enumerate(cluster_names)
]
ax.legend(handles=legend_handles, title='AE latent cluster', loc='upper left', bbox_to_anchor=(1.02, 0.95))
plt.tight_layout()

asset_png = OUTPUT_DIR / 'final_3d_asset_cluster_discovery.png'
plt.savefig(asset_png, dpi=220, bbox_inches='tight')
plt.show()

print(f'Saved static asset-cluster discovery image to: {asset_png}')
print(f'Coordinate source: {asset_coord_source}')
print(f'Weight source: {asset_weight_source}')

asset_summary = (
    asset_discovery_3d
    .groupby('LatentCluster')
    .agg(
        Names=('Weight', 'size'),
        TotalWeight=('Weight', 'sum'),
        AvgWeight=('Weight', 'mean'),
        Themes=('Theme', lambda x: ', '.join(sorted(set(map(str, x)))[:4]))
    )
    .sort_values('TotalWeight', ascending=False)
)
print('Hidden concentration summary behind the asset map:')
display(asset_summary.round(4))

try:
    import plotly.express as px
    plotly_assets = asset_discovery_3d.reset_index()
    asset_html = OUTPUT_DIR / 'interactive_3d_asset_cluster_discovery.html'
    fig_assets = px.scatter_3d(
        plotly_assets,
        x='LatentExposure1', y='LatentExposure2', z='LatentExposure3',
        color='LatentCluster',
        size='MarkerSize',
        text='Ticker',
        hover_data=['Ticker', 'Theme', 'LatentCluster', 'Weight', 'EUR_per_1000'],
        title='Interactive 3D Asset Discovery Map: Latent Clusters and Portfolio Weights'
    )
    fig_assets.update_traces(textposition='top center')
    fig_assets.write_html(asset_html)
    fig_assets.show()
    print(f'Saved interactive asset-cluster discovery map to: {asset_html}')
except Exception as e:
    print(f'Interactive Plotly asset map skipped: {e}')

### How to use these visuals in the write-up

For LinkedIn or a presentation, use the **asset-cluster discovery map** if you want to show the portfolio construction result, and use the **market-state discovery map** if you want to show the regime/stress discovery result.

The clean caption is:

> Autoencoders map hidden market structure. Random Matrix Theory cleans the risk model. Black-Litterman converts disciplined views into weights. The 3D maps show whether those discoveries are stable, clustered, and interpretable — not just backtest-friendly.